<div style="border-radius:18px;padding:34px 38px 30px;background:#0b1d3a;background-image:linear-gradient(125deg,#0b1d3a 0%,#143c74 55%,#1b6f8e 100%);color:#ffffff;font-family:system-ui,-apple-system,'Segoe UI',Roboto,sans-serif;">
<div style="font-size:12px;letter-spacing:0.22em;text-transform:uppercase;color:#cfe0f5;">CASMI 2026 &nbsp;·&nbsp; Enveda &nbsp;·&nbsp; MS/MS &rarr; structure</div>
<div style="font-size:38px;font-weight:700;line-height:1.15;margin:12px 0 10px;color:#ffffff;">From Peaks to Structures</div>
<div style="font-size:18px;line-height:1.45;color:#e4eefb;max-width:820px;">Leader-safe consensus re-ranking on top of the strongest public CASMI26 pipeline: the proven rank-1 decision is kept, the rest of each answer list listens harder to an independent spectrum&nbsp;&harr;&nbsp;structure model.</div>
<div style="margin-top:20px;font-size:13px;color:#ffffff;">
<span style="display:inline-block;padding:5px 12px;border-radius:999px;background:rgba(255,255,255,0.14);margin:0 6px 6px 0;">Base: public 0.465 pipeline, SHA-256 pinned</span>
<span style="display:inline-block;padding:5px 12px;border-radius:999px;background:rgba(255,255,255,0.14);margin:0 6px 6px 0;">Neural spectrum &harr; structure scores</span>
<span style="display:inline-block;padding:5px 12px;border-radius:999px;background:rgba(255,255,255,0.14);margin:0 6px 6px 0;">T4 GPU &middot; internet off</span>
<span style="display:inline-block;padding:5px 12px;border-radius:999px;background:rgba(98,80,214,0.55);margin:0 6px 6px 0;">Every new step fails safe</span>
</div>
</div>

<div style="font-family:system-ui,-apple-system,'Segoe UI',Roboto,sans-serif;margin-top:6px;">
<div style="display:inline-block;vertical-align:top;width:31%;min-width:240px;margin:0 1.2% 12px 0;padding:16px 18px;border-radius:14px;background:#f7f8fb;border:1px solid rgba(11,11,11,0.08);box-sizing:border-box;">
<div style="font-size:11px;letter-spacing:0.16em;text-transform:uppercase;color:#2a78d6;font-weight:600;">01 &nbsp;Same engine</div>
<div style="font-size:17px;font-weight:650;color:#0b0b0b;margin:6px 0 6px;">Byte-for-byte reproduction</div>
<div style="font-size:13.5px;line-height:1.5;color:#52514e;">The public 0.465 pipeline, with its candidate-specific PubChem promotion, runs from an embedded SHA-256-pinned copy. The spectrum&nbsp;&harr;&nbsp;structure scorer behind the 0.466 notebooks runs unchanged.</div>
</div>
<div style="display:inline-block;vertical-align:top;width:31%;min-width:240px;margin:0 1.2% 12px 0;padding:16px 18px;border-radius:14px;background:#f7f8fb;border:1px solid rgba(11,11,11,0.08);box-sizing:border-box;">
<div style="font-size:11px;letter-spacing:0.16em;text-transform:uppercase;color:#6250d6;font-weight:600;">02 &nbsp;New layer</div>
<div style="font-size:17px;font-weight:650;color:#0b0b0b;margin:6px 0 6px;">Leader-safe consensus</div>
<div style="font-size:13.5px;line-height:1.5;color:#52514e;">Rank&nbsp;1 is chosen exactly as in the 0.466 recipe. Ranks 2&ndash;25 give the neural model twice the say: that part of the list interleaves several engines, and a wrong move there costs least.</div>
</div>
<div style="display:inline-block;vertical-align:top;width:31%;min-width:240px;margin:0 0 12px 0;padding:16px 18px;border-radius:14px;background:#f7f8fb;border:1px solid rgba(11,11,11,0.08);box-sizing:border-box;">
<div style="font-size:11px;letter-spacing:0.16em;text-transform:uppercase;color:#1baf7a;font-weight:600;">03 &nbsp;Safety</div>
<div style="font-size:17px;font-weight:650;color:#0b0b0b;margin:6px 0 6px;">Fails safe, checked every run</div>
<div style="font-size:13.5px;line-height:1.5;color:#52514e;">Before any new order is trusted, the layer must rebuild the public 0.466 file exactly. Any error falls back to 0.466, then to the 0.465 answers.</div>
</div>
</div>

**Contents** &nbsp; [1 · Where the points are](#points) &nbsp;·&nbsp; [2 · The data](#data) &nbsp;·&nbsp; [3 · Pipeline map](#map) &nbsp;·&nbsp; [4 · Run](#run) &nbsp;·&nbsp; [5 · What changed](#changed) &nbsp;·&nbsp; [6 · Variants and submission plan](#plan) &nbsp;·&nbsp; [7 · Credits](#credits)

Everything you may want to change lives in the next cell.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────────────────────
VARIANT = 'V2'
#   V0  baseline 0.465 answers, no neural term
#   V1  public 0.466 recipe: neural reciprocal-rank fusion, weight 0.25 everywhere
#   V2  leader-safe consensus: rank 1 as V1, ranks 2-25 at neural weight 0.50   (default)
#   V3  V2 + tail rescue: outside candidates may take the last 3 places by neural score
#   V4  bold consensus: rank 1 as V1, ranks 2-25 at neural weight 1.00
#   'CUSTOM' uses the weights below (w_top=0 keeps the pipeline's own rank 1)
CUSTOM = dict(w_top=0.25, w_mid=0.50, rescue=0)

RRF_OFFSET = 3.0              # k in 1/(k + rank), as in the public recipe
COMMIT_MODE = 'smoke'         # 'smoke': the commit run ranks 12 visible molecules and draws every figure
                              # 'fast' : the commit run skips the pipeline (saves ~30 GPU minutes);
                              #          the scoring rerun on the hidden test always runs in full
OUTSIDERS_PER_MOLECULE = 12   # candidate pool for the tail rescue (V3, or CUSTOM with rescue > 0)
TIME_GUARD_HOURS = 7.5        # optional stages are skipped once the session has run this long
QUIET_LOGS = True             # one line per pipeline stage here; full logs stay in /kaggle/working/*.log

In [ ]:
# ── Session: paths, run mode, design tokens and small display helpers ─────────────────────────
import os, sys, re, io, json, time, math, base64, shutil, hashlib, subprocess, collections, warnings
from pathlib import Path

NB_T0 = time.time()
warnings.filterwarnings('ignore')
ROOT = Path('/kaggle/working'); ROOT.mkdir(parents=True, exist_ok=True); os.chdir(ROOT)
INPUT = Path('/kaggle/input')
CODE = ROOT / 'p2s_code'; CODE.mkdir(exist_ok=True)

import numpy as np
import pandas as pd
from IPython.display import display, HTML, FileLink

# Design tokens: chart ink and a fixed categorical order (identity never follows rank).
INK, INK2, MUTED = '#0b0b0b', '#52514e', '#898781'
GRID, AXIS, SURFACE, CARD = '#e1e0d9', '#c3c2b7', '#fcfcfb', '#f7f8fb'
BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED = (
    '#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948')
QUIET_GRAY = '#c9c8c1'
ROLE = {'main': BLUE, 'engine2': ORANGE, 'pubchem': AQUA, 'new': VIOLET}
FONT = "system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"

import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'font.size': 10.5, 'axes.titlesize': 12.5, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.titlecolor': INK, 'axes.titlepad': 10, 'axes.labelcolor': INK2, 'axes.edgecolor': AXIS,
    'axes.linewidth': 0.8, 'xtick.color': AXIS, 'ytick.color': AXIS, 'xtick.labelcolor': INK2,
    'ytick.labelcolor': INK2, 'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8,
    'grid.linestyle': '-', 'axes.axisbelow': True, 'axes.spines.top': False, 'axes.spines.right': False,
    'legend.frameon': False, 'legend.fontsize': 10, 'figure.dpi': 110, 'savefig.dpi': 160,
    'savefig.bbox': 'tight'})
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass


def tiles(items):
    """A row of stat tiles: items = [(label, value, note), ...]."""
    cells = ''.join(
        f'<div style="display:inline-block;vertical-align:top;min-width:150px;margin:0 10px 10px 0;'
        f'padding:12px 16px;border-radius:12px;background:#ffffff;border:1px solid rgba(11,11,11,0.10);">'
        f'<div style="font-size:12px;color:{INK2};">{label}</div>'
        f'<div style="font-size:25px;font-weight:650;color:{INK};line-height:1.3;">{value}</div>'
        f'<div style="font-size:11.5px;color:{MUTED};">{note}</div></div>' for label, value, note in items)
    display(HTML(f'<div style="font-family:{FONT};margin:4px 0 2px;">{cells}</div>'))


def note(title, text, color=VIOLET):
    display(HTML(f'<div style="font-family:{FONT};border-left:4px solid {color};background:{CARD};'
                 f'padding:10px 14px;border-radius:8px;margin:8px 0;font-size:13.5px;line-height:1.5;">'
                 f'<b style="color:{INK};">{title}</b> <span style="color:{INK2};">{text}</span></div>'))


def table_html(frame, highlight=None, numeric=()):
    """Plain, readable HTML table; numbers right-aligned; `highlight` = index label of a row to emphasise."""
    align = {c: 'right' if c in set(numeric) else 'left' for c in frame.columns}
    head = ''.join(f'<th style="text-align:{align[c]};padding:7px 12px;border-bottom:1px solid {AXIS};'
                   f'color:{INK2};font-weight:600;background:transparent;">{c}</th>' for c in frame.columns)
    body = []
    for idx, row in frame.iterrows():
        on = highlight is not None and idx == highlight
        style = 'background:#efedfb;font-weight:600;' if on else 'background:transparent;'
        body.append(f'<tr style="{style}">' + ''.join(
            f'<td style="text-align:{align[c]};padding:7px 12px;border-bottom:1px solid {GRID};color:{INK};'
            f'font-variant-numeric:tabular-nums;">{v}</td>' for c, v in zip(frame.columns, row.values)) + '</tr>')
    display(HTML(f'<table style="font-family:{FONT};font-size:13px;border-collapse:collapse;margin:6px 0 10px;">'
                 f'<thead><tr>{head}</tr></thead><tbody>{"".join(body)}</tbody></table>'))


def minutes(seconds):
    return f'{seconds / 60:,.1f} min' if seconds >= 60 else f'{seconds:,.0f} s'


def gpu_name():
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                             capture_output=True, text=True, timeout=30).stdout.strip()
        return ' + '.join(x.strip() for x in out.splitlines() if x.strip()) or 'none'
    except Exception:
        return 'none'


# Run mode: the commit run sees the visible format example; the scoring rerun sees the hidden test.
COMP = next((p for p in (INPUT / 'competitions/enveda-CASMI26-molecule-id-mass-spectra',
                         INPUT / 'enveda-CASMI26-molecule-id-mass-spectra',
                         INPUT / 'enveda-casmi26-molecule-id-mass-spectra')
             if all((p / f).is_file() for f in ('train.parquet', 'test.parquet', 'sample_submission.csv'))), None)
assert COMP is not None, 'Attach the competition data (enveda-CASMI26-molecule-id-mass-spectra)'
_ids = pd.read_parquet(COMP / 'test.parquet', columns=['spectrum_id']).spectrum_id.astype(str)
TEST_SIGNATURE = hashlib.md5(','.join(sorted(_ids)).encode()).hexdigest()
del _ids
IS_RERUN = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN')) or TEST_SIGNATURE != '654e030e4173780e945252b4a5adf70b'
RUN_PIPELINE = IS_RERUN or COMMIT_MODE != 'fast'
SHOW = not IS_RERUN                      # figures are drawn for readers; the scoring rerun skips them
SAMPLE_IDS = pd.read_csv(COMP / 'sample_submission.csv', dtype=str).molecule_id.tolist()
assert VARIANT in ('V0', 'V1', 'V2', 'V3', 'V4', 'CUSTOM'), VARIANT

GPU = gpu_name()
MODE = 'Scoring rerun' if IS_RERUN else ('Commit · smoke run' if RUN_PIPELINE else 'Commit · fast path')
tiles([('Run mode', MODE, 'hidden test' if IS_RERUN else 'visible format example'),
       ('Variant', VARIANT, 'set in the configuration cell'),
       ('Accelerator', GPU.split(' + ')[0] if GPU != 'none' else 'none', GPU if ' + ' in GPU else 'GPU required'),
       ('Python', '.'.join(map(str, sys.version_info[:3])), 'pipeline expects 3.12'),
       ('Answer rows', f'{len(SAMPLE_IDS):,}', 'molecules in sample_submission')])

<a id="points"></a>
## 1 · Where the points are

MRR@25 gives each molecule the reciprocal rank of its first correct structure, where *correct* means the same InChIKey connectivity block after the scorer's RDKit 2026.03.3 tautomer standardisation. With about 400 hidden molecules the public split holds roughly 125, so **one molecule is worth about 0.008 on the public board**, and a step from 0.465 to 0.466 is a fraction of one molecule.

Two facts shape the design of the new layer:

- **A rank-1 flip is worth 0.5 for its molecule**: the largest single move, and the most expensive mistake. The public pipeline spends most of its machinery on exactly this decision (library matches, PubChem promotion gates, forward models), so the layer leaves it to the rule that already scored 0.466.
- **Below rank 1 the list is an interleave** of two engines, fixed PubChem slots and a rank-2 insert. An independent model can sort that part more coherently, a wrong move there costs little, and every answer that climbs from rank 8 to rank 3 adds 0.21 for its molecule.

The right panel puts numbers on that argument; it is a model, not a measurement. The answer's baseline rank follows a distribution with MRR ≈ 0.48 and 78% recall, and the neural model adds a signal of strength μ to the answer (plus a mild preference for the pipeline's own favourites). The leader-safe rule overtakes the uniform weight once the neural model puts about 15% of the answers first among the 25 candidates. The public 0.465 → 0.466 step is too noisy to say on which side of that line the real model sits, and that is exactly what a V1-against-V2 pair of submissions measures (section 6).

In [ ]:
# ── Figure 1 · what a move is worth, and a toy model of rank fusion ───────────────────────────
def toy_fusion(mus, n=24000, seed=11, rules=((0.25, 0.25), (0.25, 0.50), (0.25, 1.00))):
    """MRR change against the baseline order for reciprocal-rank rules (w_top, w_mid), offset 3."""
    rng = np.random.default_rng(seed)
    p = np.zeros(26); p[1:6] = [0.40, 0.07, 0.035, 0.025, 0.02]
    t = np.arange(6, 26) ** -0.9; p[6:] = t / t.sum() * 0.23; p[0] = 1 - p[1:].sum()
    star = rng.choice(26, size=n, p=p)                      # 0 = answer not among the 25
    rows = np.flatnonzero(star > 0); col = star[rows] - 1
    b = np.arange(1, 26)[None, :]
    ranks = np.broadcast_to(np.arange(25), (n, 25))
    base = np.zeros(n); base[rows] = 1.0 / star[rows]
    out = {r: [] for r in rules}; top1 = []
    for mu in mus:
        z = rng.normal(size=(n, 25)) + 1.0 / np.sqrt(b)     # mild agreement with the pipeline's order
        z[rows, col] += mu
        nr = np.empty((n, 25), int); np.put_along_axis(nr, np.argsort(-z, 1, kind='stable'), ranks + 1, 1)
        top1.append(float((nr[rows, col] == 1).mean()))
        def positions(w):
            o = np.argsort(-(1 / (3 + b) + w / (3 + nr)), 1, kind='stable')
            pos = np.empty((n, 25), int); np.put_along_axis(pos, o, ranks, 1)
            return pos, o[:, 0]
        for wt, wm in rules:
            pt, lead = positions(wt)
            if wm == wt:
                final = pt[rows, col]
            else:
                pm, _ = positions(wm)
                a, l = pm[rows, col], pm[rows, lead[rows]]
                final = np.where(lead[rows] == col, 0, a + (l > a))
            mrr = np.zeros(n); mrr[rows] = 1.0 / (final + 1)
            out[(wt, wm)].append(mrr.mean() - base.mean())
    return np.array(top1), {k: np.array(v) for k, v in out.items()}


if SHOW:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.2, 4.6), gridspec_kw=dict(width_ratios=[1, 1.35], wspace=0.32))
    moves = [('rank 2 → 1', 1 - 1 / 2), ('rank 3 → 2', 1 / 2 - 1 / 3), ('rank 8 → 3', 1 / 3 - 1 / 8),
             ('rank 15 → 6', 1 / 6 - 1 / 15), ('rank 25 → 12', 1 / 12 - 1 / 25), ('outside → 25', 1 / 25)]
    y = np.arange(len(moves))[::-1]
    ax1.barh(y, [v for _, v in moves], height=0.5, color=BLUE)
    for yi, (lab, v) in zip(y, moves):
        ax1.text(v + 0.012, yi, f'+{v:.2f}', va='center', color=INK, fontsize=10)
    ax1.set_yticks(y, [m for m, _ in moves]); ax1.set_xlim(0, 0.62)
    ax1.grid(axis='y', visible=False); ax1.tick_params(axis='y', length=0)
    ax1.set_xlabel('reciprocal-rank gain for that molecule')
    ax1.set_title('What a move is worth')

    mus = np.linspace(0, 1.5, 13)
    top1, curves = toy_fusion(mus)
    names = {(0.25, 0.25): ('V1 · uniform 0.25 (public 0.466)', BLUE),
             (0.25, 0.50): ('V2 · leader-safe 0.25 / 0.50', VIOLET),
             (0.25, 1.00): ('V4 · bold 0.25 / 1.00', ORANGE)}
    ax2.axhline(0, color=AXIS, lw=1)
    for key, (lab, col) in names.items():
        ax2.plot(mus, curves[key], color=col, lw=2, solid_capstyle='round', label=lab)
        ax2.plot(mus[-1], curves[key][-1], 'o', color=col, ms=6, mec=SURFACE, mew=2)
    gap = curves[(0.25, 0.50)] - curves[(0.25, 0.25)]
    ahead = np.flatnonzero((gap[:-1] <= 0) & (gap[1:] > 0))
    if len(ahead):
        k = ahead[-1]
        cross = mus[k] + (mus[k + 1] - mus[k]) * (-gap[k]) / (gap[k + 1] - gap[k])
        hit = np.interp(cross, mus, top1)
        ax2.axvspan(cross, mus[-1] + 0.05, color=VIOLET, alpha=0.06, lw=0)
        y0, y1 = ax2.get_ylim()
        ax2.text(cross + 0.03, y0 + 0.05 * (y1 - y0), f'V2 ahead of V1 once the neural model\nranks ≥ {hit:.0%} of answers first',
                 ha='left', va='bottom', fontsize=9, color=INK2)
    ax2.set_xlim(mus[0] - 0.05, mus[-1] + 0.05)
    ax2.set_xticks(mus[::2], [f'{m:.2f}\n{t:.0%}' for m, t in zip(mus[::2], top1[::2])])
    ax2.set_xlabel('neural signal strength μ  ·  share of answers the neural model ranks first')
    ax2.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f'{v:+.3f}'))
    ax2.set_ylabel('MRR@25 change vs baseline order')
    ax2.set_title('Toy model: leader-safe fusion gains when the neural model is informative')
    ax2.legend(loc='upper left')
    plt.show()

<a id="data"></a>
## 2 · The data in one screen

Each molecule arrives as 1 to 16 MS/MS spectra: fragment m/z values with normalised intensities, the precursor m/z, the ion form (adduct), polarity and collision energy. The answer is one ordered list of up to 25 SMILES.

The visible `test.parquet` is a format example drawn from the training spectra. The scoring rerun swaps in the hidden test: natural products from biological extracts measured on timsTOF instruments, 157 to 1,159 Da. The panels below describe the visible file only, so they show the format of the task, not the hidden distribution.

In [ ]:
# ── Figure 2 · the visible test file at a glance ─────────────────────────────────────────────
def mean_ce(v):
    try:
        a = np.atleast_1d(np.asarray(v, dtype=float))
        return float(np.nanmean(a)) if a.size else np.nan
    except (TypeError, ValueError):
        return np.nan


if SHOW:
    try:
        te_view = pd.read_parquet(COMP / 'test.parquet')
        te_view['ce'] = te_view.collision_energy_ev.map(mean_ce) if 'collision_energy_ev' in te_view else np.nan
        te_view['n_peaks'] = te_view.ms2_mzs.map(len)
        per_mol = te_view.groupby('molecule_id').size()
        tiles([('Molecules', f'{te_view.molecule_id.nunique():,}', 'visible format example'),
               ('Spectra', f'{len(te_view):,}', f'median {per_mol.median():.0f} per molecule'),
               ('Ion forms', f'{te_view.adduct.nunique()}', f'most common {te_view.adduct.mode().iat[0]}'),
               ('Peaks per spectrum', f'{te_view.n_peaks.median():.0f}', 'median'),
               ('Positive mode', f"{te_view.ionization_mode.eq('positive').mean():.0%}", 'share of spectra')])

        fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), gridspec_kw=dict(wspace=0.38))
        counts = per_mol.value_counts().sort_index()
        axes[0].bar(counts.index, counts.values, width=0.62, color=BLUE)
        axes[0].set_title('Spectra per molecule'); axes[0].set_xlabel('spectra'); axes[0].set_ylabel('molecules')
        axes[0].grid(axis='x', visible=False)

        share = te_view.adduct.value_counts(normalize=True)
        top = share.head(5)
        if len(share) > 5:
            top = pd.concat([top, pd.Series({'other': share.iloc[5:].sum()})])
        yy = np.arange(len(top))[::-1]
        axes[1].barh(yy, top.values, height=0.55, color=BLUE)
        for yi, v in zip(yy, top.values):
            axes[1].text(v + 0.01, yi, f'{v:.0%}', va='center', fontsize=9.5, color=INK)
        axes[1].set_yticks(yy, top.index); axes[1].set_xlim(0, min(1.0, top.max() * 1.35))
        axes[1].xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
        axes[1].grid(axis='y', visible=False); axes[1].tick_params(axis='y', length=0)
        axes[1].set_title('Ion form (adduct)')

        ce = te_view.ce.dropna()
        if len(ce):
            axes[2].hist(ce, bins=24, color=BLUE, rwidth=0.85)
        axes[2].set_title('Collision energy'); axes[2].set_xlabel('eV'); axes[2].set_ylabel('spectra')
        axes[2].grid(axis='x', visible=False)

        axes[3].hist(te_view.precursor_mz, bins=28, color=BLUE, rwidth=0.85)
        axes[3].set_title('Precursor m/z'); axes[3].set_xlabel('m/z'); axes[3].set_ylabel('spectra')
        axes[3].grid(axis='x', visible=False)
        plt.show()

        # One molecule, all of its spectra: stems scaled to the base peak, precursor marked.
        example_mid = per_mol.sort_values(ascending=False, kind='stable').index[0]
        ex = te_view[te_view.molecule_id == example_mid].sort_values('ce', kind='stable').head(4)
        fig, axes = plt.subplots(len(ex), 1, figsize=(13, 1.55 * len(ex) + 0.6), sharex=True,
                                 gridspec_kw=dict(hspace=0.25))
        axes = np.atleast_1d(axes)
        xmax = float(ex.precursor_mz.max()) * 1.06
        for ax, r in zip(axes, ex.itertuples(index=False)):
            mz = np.asarray(r.ms2_mzs, float); it = np.asarray(r.ms2_normalized_intensities, float)
            it = it / it.max() if it.size and it.max() > 0 else it
            ax.vlines(mz, 0, it, color=BLUE, lw=1.1)
            ax.plot([r.precursor_mz], [1.08], 'v', color=ORANGE, ms=6)
            ax.set_ylim(0, 1.42); ax.set_yticks([0, 1], ['0', '1']); ax.grid(axis='x', visible=False)
            ce_txt = f'{r.ce:.0f} eV' if np.isfinite(r.ce) else 'CE n/a'
            ax.text(0.003, 0.97, f'{r.adduct} · {ce_txt} · {len(mz)} peaks', transform=ax.transAxes,
                    fontsize=9.5, color=INK2, va='top')
        axes[0].set_title(f'One molecule, every spectrum: {example_mid}  (▼ precursor)')
        axes[-1].set_xlabel('m/z'); axes[-1].set_xlim(0, xmax)
        plt.show()
        del te_view
    except Exception as exc:
        print('data overview skipped:', type(exc).__name__, exc)

<a id="map"></a>
## 3 · Pipeline map

The first three columns are the public pipeline, assembled over many public versions by many authors (credited in section 7) and executed here unchanged. The fourth column is this notebook.

- **Candidates come from three places.** Engine 1 searches 0.71 M stored structures (training set and COCONUT) within 10 ppm of the neutral mass and generates analogues of spectrally similar library compounds. Engine 2 adds ChEBI, LIPID MAPS and PubChemLite. The PubChem channel screens 105.9 M PubChem structures outside the pool with fingerprint models and a documentation prior.
- **Fusion is identity-aware.** Structures that the scorer would count as the same answer get one vote; GLACIER, ICEBERG or a MetFrag-lite fragment score re-orders same-formula candidates among the best 40.
- **The new layer only re-orders the final list**, so the candidate set and every upstream decision stay as published. The optional tail rescue (V3) is the one exception: it may swap the last three places for candidates the pipeline had ranked just outside its list.

In [ ]:
# ── Figure 3 · pipeline map ──────────────────────────────────────────────────────────────────
from matplotlib.patches import FancyBboxPatch


def pipeline_map():
    fig = plt.figure(figsize=(14.5, 6.4))
    ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
    cols = [
        ('Evidence', None, [
            ('MS/MS spectra', '1–16 per molecule · timsTOF\nadduct, polarity, collision energy', MUTED),
            ('Reference library', '2.54 M training spectra\n+ 751 public MassBank / MoNA / GNPS', MUTED),
            ('Candidate stores', 'pool 0.71 M (training + COCONUT)\nPubChem tier 105.9 M + supplements\nChEBI · LIPID MAPS · PubChemLite', MUTED)]),
        ('Candidate engines', None, [
            ('Engine 1', 'FPNet full1 · 4 LightGBM rankers\nanalogue generation · popularity re-order', BLUE),
            ('Engine 2', 'two rankers (BIO + AFIX)\n8 + 4 gradient-boosted models', ORANGE),
            ('PubChem channel', 'FPNet A + B + fold-safe model\ndocumentation prior · 25 proposals', AQUA)]),
        ('Fusion and physics', None, [
            ('Gated merge', 'PubChem slots when library match < 0.9\nrank-1 promotion on the leader\'s evidence', MUTED),
            ('Identity fusion', 'one vote per metric identity\nweights 1.0 / 0.6, offset 3', MUTED),
            ('Forward models', 'GLACIER · ICEBERG · MetFrag-lite\nsame-formula re-order of the best 40', MUTED),
            ('Rank-2 placement', 'Engine 2 leader at rank 2', MUTED)]),
        ('This notebook', VIOLET, [
            ('Neural scores', 'spectrum ↔ structure model\n(unchanged public scorer)', VIOLET),
            ('Leader-safe consensus', 'rank 1: weight 0.25 (public rule)\nranks 2–25: weight 0.50', VIOLET),
            ('Tail rescue · optional', 'outside candidates compete\nfor the last 3 places', VIOLET),
            ('Fail-safe selector', 'must rebuild the 0.466 file exactly\n→ submission.csv', VIOLET)]),
    ]
    x0s, w = [0.012, 0.264, 0.516, 0.768], 0.22
    top, bottom = 0.885, 0.04
    for (head, head_col, boxes), x0 in zip(cols, x0s):
        ax.text(x0 + 0.004, 0.955, head.upper(), fontsize=10, fontweight='bold',
                color=head_col or INK2, va='center')
        gap = 0.022
        h = (top - bottom - gap * (len(boxes) - 1)) / len(boxes)
        for i, (title, body, stripe) in enumerate(boxes):
            y = top - (i + 1) * h - i * gap
            new = head_col is not None
            ax.add_patch(FancyBboxPatch((x0, y), w, h, boxstyle='round,pad=0,rounding_size=0.012',
                                        fc='#f1effc' if new else '#ffffff', ec='#d8d3f6' if new else '#dddcd5', lw=1.0))
            ax.add_patch(FancyBboxPatch((x0, y + 0.012), 0.005, h - 0.024, boxstyle='round,pad=0,rounding_size=0.002',
                                        fc=stripe, ec='none'))
            ax.text(x0 + 0.017, y + h - 0.03, title, fontsize=12, fontweight='bold', color=INK, va='top')
            ax.text(x0 + 0.017, y + h - 0.08, body, fontsize=10, color=INK2, va='top', linespacing=1.5)
    for xa in (0.236, 0.488, 0.740):
        ax.annotate('', xy=(xa + 0.024, 0.46), xytext=(xa + 0.002, 0.46),
                    arrowprops=dict(arrowstyle='-|>', color=AXIS, lw=1.6, mutation_scale=16))
    plt.show()


if SHOW:
    pipeline_map()

<a id="run"></a>
## 4 · Run

Four stages, each in its own process so GPU memory is released between them. The commit run ranks the first 12 visible molecules and fills the other rows with placeholders, which takes about 30 GPU minutes; the scoring rerun on the hidden test ranks every molecule.

| Stage | What runs | If it fails |
|---|---|---|
| 0 · Sources | Writes the pinned pipeline notebook and helper modules, checks their SHA-256, finds the neural model bundle, installs the official RDKit 2026.03.3 wheel | The run stops: an input is missing |
| 1 · Pipeline | The public 0.465 pipeline, every cell in order, then an export of its final lists | The run stops, as in the public notebooks |
| 2 · Neural scores | The public spectrum&nbsp;&harr;&nbsp;structure scorer on the final candidates (and, for V3, on outside candidates) | The 0.465 answers are submitted |
| 3 · Consensus | Builds every variant, proves that V1 equals the public 0.466 file, validates the chosen variant | Falls back to V1, V0, then the public 0.466 file |

The next cell holds the pinned sources as text (about 0.5 MB, hidden in the viewer). The public pipeline notebook is embedded byte for byte as base64; the six helper modules of the 0.466 notebooks are embedded unchanged; `consensus.py` and `outsiders.py` are this notebook's additions.

In [ ]:
# ── Pinned sources (generated; hidden in the viewer) ─────────────────────────────────────────
# MODULES: the six helper modules of the public 0.466 notebooks, unchanged, plus this notebook's
# consensus.py and outsiders.py. FROZEN_B64: the public 0.465 pipeline notebook, byte for byte.
MODULES = {}
MODULES['runner.py'] = '"""Execute the pinned corrected baseline once, then export its final candidates."""\nimport os,sys,json,hashlib\nfrom pathlib import Path\nfrom IPython.display import display\nif __name__==\'__main__\':\n    source=Path(sys.argv[1]); expected=sys.argv[2]; root=Path(sys.argv[3]).resolve()\n    assert hashlib.sha256(source.read_bytes()).hexdigest()==expected, \'Source hash differs\'\n    assert not os.environ.get(\'INC_PANEL\'), \'Replay input is not allowed\'\n    root.mkdir(parents=True,exist_ok=True); os.chdir(root)\n    assert not (root/\'work/spec_meta.parquet\').exists(), \'Fresh pipeline directory required\'\n    assert not (root/\'submission.csv\').exists(), \'Fresh pipeline directory required\'\n    scope={\'__name__\':\'__main__\',\'display\':display}\n    for i,c in enumerate(json.loads(source.read_text())[\'cells\']):\n        if c[\'cell_type\']!=\'code\': continue\n        print(f\'PIPELINE CELL {i} START\',flush=True)\n        exec(compile(\'\'.join(c[\'source\']),f\'pipeline_cell_{i}\',\'exec\'),scope)\n        print(f\'PIPELINE CELL {i} DONE\',flush=True)\n    assert \'candidate_promotion_evidence\' in scope, \'Candidate-specific promotion fix missing\'\n    from export_evidence import export\n    export(scope,root)\n    (root/\'execution_receipt.json\').write_text(json.dumps(dict(source_sha256=expected,\n        mode=\'submission\',source_adaptations=[],candidate_promotion_fix=True,\n        is_full_run=bool(scope[\'IS_RERUN\']),processed_molecules=int(scope[\'te\'].molecule_id.nunique())),indent=2))\n'
MODULES['export_evidence.py'] = '"""Executed in the frozen pipeline process AFTER every original final rule."""\nimport math,json,shutil\nfrom pathlib import Path\nimport pandas as pd\n\ndef export(scope, root):\n    root=Path(root);frame=pd.read_csv(root/\'submission.csv\',dtype=str,keep_default_na=False)\n    shutil.copy2(root/\'submission.csv\',root/\'submission_baseline0465.csv\')\n    processed=set(scope[\'te\'].molecule_id.astype(str));identity=scope[\'metric_identity\']\n    def rank_map(smis):\n        out={}\n        for rank,s in enumerate(smis,1):\n            key=identity(s)\n            if key is not None:out.setdefault(key,rank)\n        return out\n    def finite(v):\n        try:return float(v) if v is not None and math.isfinite(float(v)) else None\n        except (ValueError,TypeError):return None\n    rows=[]\n    for mid,answer in frame.itertuples(index=False):\n        candidates=answer.split(\';\')\n        assert 1<=len(candidates)<=25 and len(candidates)==len(set(candidates)),mid\n        active=mid in processed\n        base=scope[\'BASE\'].get(mid,([],[],0.))\n        maps={\n            \'main\':rank_map(base[0]),\n            \'prefusion\':rank_map(scope.get(\'PREFUSION_LISTS\',{}).get(mid,[])),\n            \'engine2\':rank_map(scope.get(\'ENG\',{}).get(mid,{}).get(\'smiles\',[])),\n            \'pubchem\':rank_map(scope.get(\'PC\',{}).get(mid,{}).get(\'pc\',[]))}\n        ice=scope.get(\'ICE_SCORES\',{}).get(mid,{}) or {}\n        gl=scope[\'gl_of\'](mid) or {}\n        for rank,s in enumerate(candidates,1):\n            ident=identity(s);key=scope[\'chem\'].score_key(s)\n            row=dict(molecule_id=mid,smiles=s,metric_key=ident[0] if ident else None,baseline_rank=rank,\n                processed=active,library_max=finite(base[2]) or 0.,proxy_gl=mid in scope.get(\'N1_GL_MIDS\',set()),\n                ice=finite(ice.get(s,ice.get(key))),gl=finite(gl.get(s,gl.get(key))))\n            row.update({name+\'_rank\':mapping.get(ident,0) for name,mapping in maps.items()});rows.append(row)\n    pd.DataFrame(rows).to_parquet(root/\'frozen_candidate_evidence.parquet\',index=False)\n    # Save the effective query spectra used by the pipeline (12 rows\' molecules in visible smoke, all in rerun).\n    scope[\'te\'].to_parquet(root/\'effective_query_spectra.parquet\',index=False)\n    (root/\'frozen_run_info.json\').write_text(json.dumps(dict(processed_molecules=len(processed),\n        output_molecules=len(frame),is_full_run=bool(scope[\'IS_RERUN\']),\n        note=\'Baseline file saved after the original Engine2 rank-2 insertion; candidate-specific PubChem promotion fix included.\'),indent=2))\n    print(\'EXPORTED FROZEN FINAL25\',len(rows),\'candidate rows\',flush=True)\n'
MODULES['neural_scores.py'] = '"""Runs after the public pipeline process exits, freeing its GPU memory."""\nimport json,sys,importlib.util,types,hashlib\nfrom pathlib import Path\nimport numpy as np,pandas as pd,torch\n\ndef load_model(bundle,checkpoint,device):\n    # Private module namespace prevents collision with public engines\' model/features modules.\n    package=\'_casmi_own\';p=types.ModuleType(package);p.__path__=[str(bundle)];sys.modules[package]=p\n    for name in [\'features\',\'model\']:\n        spec=importlib.util.spec_from_file_location(package+\'.\'+name,Path(bundle)/(name+\'.py\'))\n        mod=importlib.util.module_from_spec(spec);sys.modules[spec.name]=mod;spec.loader.exec_module(mod)\n    features=sys.modules[package+\'.features\'];cls=sys.modules[package+\'.model\'].SpectrumCandidateModel\n    cfg=json.loads((Path(bundle)/\'config.json\').read_text());model=cls(cfg).to(device)\n    ck=torch.load(Path(bundle)/checkpoint,map_location=device,weights_only=True)\n    model.load_state_dict(ck[\'state_dict\']);model.eval()\n    return model,features,cfg\n\n@torch.inference_mode()\ndef score(bundle,checkpoint,evidence,queries,output,device=\'cuda\'):\n    torch.set_num_threads(4)\n    if device==\'cuda\':assert torch.cuda.is_available(),\'GPU required for neural scoring\'\n    model,feat,cfg=load_model(bundle,checkpoint,device)\n    frame=pd.read_parquet(evidence);q=pd.read_parquet(queries);q[\'molecule_id\']=q.molecule_id.astype(str)\n    query={mid:g for mid,g in q.groupby(\'molecule_id\',sort=False)};cache={};parts=[];errors=[]\n    for index,(mid,g) in enumerate(frame.groupby(\'molecule_id\',sort=False)):\n        g=g.sort_values(\'baseline_rank\',kind=\'stable\').copy()\n        g[\'neural_ok\']=False\n        if not g.processed.iloc[0]:\n            for c in [\'neural_cos\',\'neural_fp\',\'neural_score\',\'neural_std\',\'n_spectra\',\'n_adducts\']:g[c]=0.\n            parts.append(g);continue\n        try:\n            assert mid in query,mid\n            records=[]\n            # Use EVERY observed query spectrum; no adduct merging and no new spectrum selection rule.\n            for row in query[mid].itertuples(index=False):\n                try:\n                    pk,n=feat.peaks(row.ms2_mzs,row.ms2_normalized_intensities,row.precursor_mz,cfg[\'max_peaks\'])\n                    if n<cfg[\'min_peaks\']:continue\n                    m,a=feat.metadata(row.precursor_mz,row.adduct,row.ionization_mode,row.collision_energy_ev)\n                    records.append((pk,n,m,a))\n                except (ValueError,TypeError):continue\n            if not records:raise ValueError(\'No usable spectra after the training preprocessing filters\')\n            fp=[]\n            for s in g.smiles:\n                if s not in cache:cache[s]=feat.structure(s,cfg[\'fp_bits\'])[-1]\n                fp.append(cache[s])\n            F=torch.tensor(np.stack(fp),device=device,dtype=torch.float32);M=model.encode_molecules(F)\n            cos_all=[];fp_all=[]\n            for st in range(0,len(records),64):\n                b=records[st:st+64]\n                x=torch.tensor(np.stack([v[0] for v in b]),device=device)\n                lengths=torch.tensor([v[1] for v in b],device=device,dtype=torch.long)\n                meta=torch.tensor(np.stack([v[2] for v in b]),device=device)\n                adduct=torch.tensor([v[3] for v in b],device=device,dtype=torch.long)\n                Z,P=model.encode_spectra(x,lengths,meta,adduct)\n                cos_all.append((Z@M.T).cpu().numpy());fp_all.append((P@F.T/cfg[\'fp_bits\']).cpu().numpy())\n            cos=np.concatenate(cos_all);fps=np.concatenate(fp_all)\n            combined=cos/cfg[\'temperature\']+cfg[\'fp_score_weight\']*fps\n            assert np.isfinite(combined).all(),mid\n            g[\'neural_cos\']=cos.mean(0);g[\'neural_fp\']=fps.mean(0);g[\'neural_score\']=combined.mean(0)\n            g[\'neural_std\']=combined.std(0);g[\'n_spectra\']=len(records);g[\'n_adducts\']=query[mid].adduct.nunique()\n            g[\'neural_ok\']=True\n            parts.append(g)\n        except (ValueError,TypeError) as exc:\n            errors.append(dict(molecule_id=mid,error=str(exc)))\n            for col in [\'neural_cos\',\'neural_fp\',\'neural_score\',\'neural_std\',\'n_spectra\',\'n_adducts\']:g[col]=0.\n            parts.append(g)\n        if (index+1)%25==0:print(\'NEURAL SCORED\',index+1,\'molecules\',flush=True)\n    result=pd.concat(parts,ignore_index=True)\n    assert len(result)==len(frame)\n    result.to_parquet(output,index=False)\n    Path(str(output)+\'.errors.json\').write_text(json.dumps(errors,indent=2))\n    print(\'NEURAL input failures kept at baseline:\',len(errors),flush=True)\n    print(\'NEURAL FEATURES\',result.shape,checkpoint,flush=True)\n\nif __name__==\'__main__\':score(*sys.argv[1:])\n'
MODULES['submission.py'] = 'import json,hashlib\nfrom pathlib import Path\nimport pandas as pd,numpy as np\nfrom ranking import rrf_order\n\ndef apply(evidence,baseline,output,weight=.25,offset=3.):\n    f=pd.read_parquet(evidence);base=pd.read_csv(baseline,dtype=str,keep_default_na=False)\n    groups={mid:g for mid,g in f.groupby(\'molecule_id\',sort=False)};rows=[];audit=[]\n    assert base.molecule_id.is_unique and set(groups)==set(base.molecule_id)\n    for mid,text in base.itertuples(index=False):\n        g=groups[mid].sort_values(\'baseline_rank\',kind=\'stable\');original=text.split(\';\')\n        assert g.smiles.tolist()==original,mid\n        active=bool(g.processed.iloc[0]) and bool(g.neural_ok.all())\n        order=rrf_order(g.neural_score.values,weight,offset) if active else np.arange(len(g))\n        new=g.smiles.iloc[order].tolist()\n        assert len(new)==len(original) and set(new)==set(original),mid\n        assert len(new)<=25 and len(set(new))==len(new)\n        rows.append((mid,\';\'.join(new)))\n        audit.append(dict(molecule_id=mid,processed=active,changed=new!=original,\n            changed_top1=new[0]!=original[0],candidate_count=len(new),weight=weight))\n    result=pd.DataFrame(rows,columns=[\'molecule_id\',\'smiles\'])\n    assert result.molecule_id.tolist()==base.molecule_id.tolist()\n    result.to_csv(output,index=False)\n    out=Path(output).parent;pd.DataFrame(audit).to_csv(out/\'neural_rerank_audit.csv\',index=False)\n    report=dict(method=\'baseline rank + neural rank reciprocal-rank fusion AFTER every baseline rule\',\n        neural_weight=weight,rrf_offset=offset,weight_is_validated=False, validation_scope="Fixed-weight submission experiment; no validation performed",\n        candidate_set_preserved=True,changed_rows=sum(x[\'changed\'] for x in audit),\n        changed_top1=sum(x[\'changed_top1\'] for x in audit),rows=len(rows),\n        frozen_sha256=hashlib.sha256(Path(baseline).read_bytes()).hexdigest(),\n        experimental_sha256=hashlib.sha256(Path(output).read_bytes()).hexdigest())\n    (out/\'neural_submission_report.json\').write_text(json.dumps(report,indent=2));print(json.dumps(report,indent=2))\n    return report\n'
MODULES['ranking.py'] = '"""Pure deterministic ranking utilities shared by both notebooks."""\nimport numpy as np\nimport pandas as pd\n\ndef stable_rank(values):\n    values=np.asarray(values,dtype=float)\n    assert np.isfinite(values).all(), \'Nonfinite candidate scores\'\n    order=np.argsort(-values,kind=\'stable\');ranks=np.empty(len(values),int);ranks[order]=np.arange(1,len(values)+1)\n    return ranks\n\ndef rrf_order(neural_scores,weight=0.15,offset=3.0):\n    assert weight>=0 and offset>=0\n    n=len(neural_scores);baseline=np.arange(1,n+1)\n    # Zero weight guarantees exact baseline order even when neural scores are unavailable.\n    if weight==0:return np.arange(n)\n    ranks=stable_rank(neural_scores)\n    return np.argsort(-(1/(offset+baseline)+weight/(offset+ranks)),kind=\'stable\')\n\ndef metrics(frame,score=\'prediction\'):\n    out=[]\n    for mid,g in frame.groupby(\'molecule_id\',sort=False):\n        g=g.sort_values(\'baseline_rank\',kind=\'stable\')\n        order=np.argsort(-g[score].to_numpy(),kind=\'stable\')\n        labels=g.label.to_numpy()[order];where=np.flatnonzero(labels)\n        r=int(where[0]+1) if len(where) else None\n        out.append(dict(molecule_id=mid,mrr25=1/r if r and r<=25 else 0.,top1=int(r==1),\n                        candidate_recall=int(bool(len(where))),rank=r))\n    return pd.DataFrame(out)\n\nBASE_FEATURES=[\'baseline_rr\',\'main_rr\',\'prefusion_rr\',\'engine2_rr\',\'pubchem_rr\',\'engine_agreement\',\n    \'library_max\',\'proxy_gl\',\'ice_missing\',\'gl_missing\',\'ice_z\',\'gl_z\',\n    \'neural_cos_z\',\'neural_fp_z\',\'neural_score_z\',\'neural_rr\',\'neural_std\',\n    \'n_spectra_log\',\'n_adducts\',\'neural_top_gap\',\'neural_agrees_top1\',\'engine2_agrees_top1\',\n    \'neural_delta_rank\',\'neural_x_library\',\'neural_x_agreement\',\'neural_x_sim_missing\',\n    \'main_missing\',\'engine2_missing\',\'pubchem_missing\']\n\ndef make_features(frame):\n    f=frame.copy()\n    for src in [\'baseline\',\'main\',\'prefusion\',\'engine2\',\'pubchem\']:\n        r=pd.to_numeric(f[src+\'_rank\'],errors=\'coerce\');f[src+\'_rr\']=np.where(r>0,1/(3+r),0)\n        if src in [\'main\',\'engine2\',\'pubchem\']:f[src+\'_missing\']=(~(r>0)).astype(float)\n    for src in [\'ice\',\'gl\',\'neural_cos\',\'neural_fp\',\'neural_score\']:\n        v=pd.to_numeric(f[src],errors=\'coerce\');v=v.where(np.isfinite(v))\n        if src in [\'ice\',\'gl\']:f[src+\'_missing\']=v.isna().astype(float)\n        grp=v.groupby(f.molecule_id,sort=False)\n        f[src+\'_z\']=((v-grp.transform(\'mean\'))/grp.transform(lambda s:s.std(ddof=0)).clip(lower=1e-6)).fillna(0).clip(-5,5)\n    f[\'neural_rank\']=0.;f[\'neural_top_gap\']=0.;f[\'neural_agrees_top1\']=0.;f[\'engine2_agrees_top1\']=0.\n    for _,g in f.groupby(\'molecule_id\',sort=False):\n        idx=g.index;rank=stable_rank(g.neural_score.values);f.loc[idx,\'neural_rank\']=rank\n        vals=np.sort(g.neural_score_z.values)[::-1]\n        f.loc[idx,\'neural_top_gap\']=float(vals[0]-vals[1]) if len(vals)>1 else 0.\n        f.loc[idx,\'neural_agrees_top1\']=float(g.iloc[np.argmin(rank)].baseline_rank==1)\n        f.loc[idx,\'engine2_agrees_top1\']=float(((g.baseline_rank==1)&(g.engine2_rank==1)).any())\n    f[\'neural_rr\']=1/(3+f.neural_rank)\n    f[\'engine_agreement\']=((f.main_rank>0).astype(int)+(f.engine2_rank>0).astype(int)+(f.pubchem_rank>0).astype(int))\n    f[\'n_spectra_log\']=np.log1p(f.n_spectra)\n    f[\'neural_delta_rank\']=(f.baseline_rank-f.neural_rank)/25\n    f[\'neural_x_library\']=f.neural_score_z*f.library_max\n    f[\'neural_x_agreement\']=f.neural_score_z*f.engine_agreement\n    f[\'neural_x_sim_missing\']=f.neural_score_z*f.ice_missing*f.gl_missing\n    X=f[BASE_FEATURES].astype(float).replace([np.inf,-np.inf],np.nan).fillna(0.)\n    assert np.isfinite(X.values).all()\n    return X\n\ndef residual_scores(frame,probability,beta):\n    p=np.clip(np.asarray(probability,float),1e-5,1-1e-5)\n    raw=pd.Series(np.log(p/(1-p)),index=frame.index)\n    grp=raw.groupby(frame.molecule_id,sort=False)\n    z=((raw-grp.transform(\'mean\'))/grp.transform(lambda s:s.std(ddof=0)).clip(lower=1e-6)).fillna(0).clip(-5,5)\n    correction=float(beta)*z.values\n    if \'neural_ok\' in frame:correction=np.where(frame.neural_ok.values,correction,0.)\n    return -np.log(frame.baseline_rank.values.astype(float))+correction\n'
MODULES['bundle.py'] = "import hashlib,json,zipfile\nfrom pathlib import Path\n\ndef sha(path):\n    h=hashlib.sha256()\n    with open(path,'rb') as f:\n        for b in iter(lambda:f.read(8<<20),b''):h.update(b)\n    return h.hexdigest()\n\ndef find_bundle(input_root,output_root,pins):\n    roots=[]\n    for path in Path(input_root).rglob('final_model.pt'):\n        if (path.parent/'config.json').is_file():roots.append(path.parent)\n    if not roots:\n        archives=list(Path(input_root).rglob('casmi_spectrum_model_v1.zip'))\n        assert len(archives)==1, 'Attach the supplied casmi_spectrum_model_v1.zip as a Kaggle dataset (zipped or extracted)'\n        dest=Path(output_root)/'uploaded_model';dest.mkdir(parents=True,exist_ok=True)\n        with zipfile.ZipFile(archives[0]) as z:\n            for item in z.infolist():\n                target=(dest/item.filename).resolve()\n                assert target.is_relative_to(dest.resolve()),'Unsafe ZIP member'\n            z.extractall(dest)\n        roots=[p.parent for p in dest.rglob('final_model.pt')]\n    matching=[r for r in roots if all((r/n).is_file() and sha(r/n)==h for n,h in pins.items())]\n    assert len(matching)==1, f'Expected exactly one matching model bundle, found {len(matching)}. Use the attached v1 artifacts.'\n    return matching[0]\n"
MODULES['consensus.py'] = '"""Leader-safe consensus re-ranking for CASMI26 answer lists.\n\nRuns after the frozen 0.465 pipeline and after the neural scorer. It never reruns a\nmodel: it reorders the candidates of each molecule and, only when asked, lets a few\ncandidates that the pipeline left outside its list compete for the last places.\n\nOrder rule\n    rank 1   = leader of   1/(k + b) + w_top / (k + n)\n    ranks 2+ = the others, ordered by   1/(k + b) + w_mid / (k + n)\nwith b the baseline rank, n the neural rank among the same candidates and k the\noffset. With w_top == w_mid this is exactly the fixed-weight reciprocal-rank fusion\nof the public 0.466 notebook (``ranking.rrf_order``), which build_variants re-checks.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom ranking import rrf_order, stable_rank\n\nPRESETS = {\n    \'V0\': dict(w_top=0.00, w_mid=0.00, rescue=0, label=\'Baseline 0.465 - no neural term\'),\n    \'V1\': dict(w_top=0.25, w_mid=0.25, rescue=0, label=\'Neural RRF 0.25 - public 0.466 recipe\'),\n    \'V2\': dict(w_top=0.25, w_mid=0.50, rescue=0, label=\'Leader-safe consensus 0.25 / 0.50\'),\n    \'V3\': dict(w_top=0.25, w_mid=0.50, rescue=3, label=\'Leader-safe consensus + 3-slot tail rescue\'),\n    \'V4\': dict(w_top=0.25, w_mid=1.00, rescue=0, label=\'Bold consensus 0.25 / 1.00\'),\n}\n\n\ndef sha256(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef leader_safe_order(neural_scores, w_top, w_mid, offset=3.0):\n    """Indices of the new order (baseline order = 0..n-1)."""\n    s = np.asarray(neural_scores, dtype=float)\n    if len(s) == 0:\n        return np.zeros(0, dtype=int)\n    first = np.asarray(rrf_order(s, w_top, offset), dtype=int)\n    if w_mid == w_top:\n        return first\n    lead = int(first[0])\n    rest = [int(i) for i in rrf_order(s, w_mid, offset) if int(i) != lead]\n    return np.asarray([lead] + rest, dtype=int)\n\n\ndef rescue_tail(smiles, scores, outsiders, slots, limit=25):\n    """Let outside candidates compete for the last `slots` places by neural score.\n\n    smiles, scores: the reordered list and its neural scores.\n    outsiders: [(smiles, neural_score)] whose metric identities differ from every\n    listed candidate and from each other (guaranteed by outsiders.py).\n    Free places of a list shorter than `limit` are offered as well.\n    Returns (new_list, number_of_outsiders_admitted).\n    """\n    smiles = list(smiles)\n    scores = [float(v) for v in scores]\n    if slots <= 0 or not outsiders or not smiles:\n        return smiles, 0\n    k = min(int(slots), len(smiles))\n    free = max(0, int(limit) - len(smiles))\n    head = smiles[:len(smiles) - k]\n    pool = [(s, v, 0) for s, v in zip(smiles[len(smiles) - k:], scores[len(scores) - k:])]\n    pool += [(str(s), float(v), 1) for s, v in outsiders if np.isfinite(float(v)) and str(s) not in smiles]\n    pool.sort(key=lambda t: (-t[1], t[2]))  # stable; on ties a listed candidate keeps its place\n    chosen = pool[:k + free]\n    out = (head + [s for s, _, _ in chosen])[:limit]\n    return out, int(sum(t[2] for t in chosen))\n\n\ndef read_outsiders(path):\n    out = {}\n    if not path or not Path(path).is_file():\n        return out\n    frame = pd.read_parquet(path)\n    if frame.empty or \'neural_ok\' not in frame.columns:\n        return out\n    score = pd.to_numeric(frame.neural_score, errors=\'coerce\')\n    frame = frame[frame.neural_ok.astype(bool) & np.isfinite(score)]\n    for mid, g in frame.groupby(\'molecule_id\', sort=False):\n        g = g.sort_values(\'baseline_rank\', kind=\'stable\')\n        out[str(mid)] = list(zip(g.smiles.astype(str), g.neural_score.astype(float)))\n    return out\n\n\ndef build_variants(evidence, baseline, outdir, presets=None, offset=3.0, outsiders=None):\n    """Write submission_<name>.csv for every preset; return (report, moves)."""\n    presets = dict(PRESETS if presets is None else presets)\n    frame = pd.read_parquet(evidence)\n    base = pd.read_csv(baseline, dtype=str, keep_default_na=False)\n    assert list(base.columns) == [\'molecule_id\', \'smiles\'] and base.molecule_id.is_unique\n    groups = {str(m): g.sort_values(\'baseline_rank\', kind=\'stable\') for m, g in frame.groupby(\'molecule_id\', sort=False)}\n    assert set(groups) == set(base.molecule_id), \'evidence and baseline cover different molecules\'\n    outs = read_outsiders(outsiders)\n    outdir = Path(outdir)\n    outdir.mkdir(parents=True, exist_ok=True)\n    report, moves = {}, []\n    for name, cfg in presets.items():\n        rows = []\n        stats = dict(active_rows=0, changed_rows=0, changed_top1=0, rescued=0)\n        for mid, text in base.itertuples(index=False):\n            original = text.split(\';\')\n            g = groups[mid]\n            assert g.smiles.tolist() == original, f\'evidence order differs from the baseline: {mid}\'\n            active = bool(g.processed.iloc[0]) and bool(g.neural_ok.all())\n            if active:\n                order = leader_safe_order(g.neural_score.values, cfg[\'w_top\'], cfg[\'w_mid\'], offset)\n            else:\n                order = np.arange(len(g))\n            new = g.smiles.iloc[order].tolist()\n            new_scores = g.neural_score.astype(float).values[order].tolist()\n            rescued = 0\n            if active and int(cfg.get(\'rescue\', 0)) > 0 and outs.get(mid):\n                new, rescued = rescue_tail(new, new_scores, outs[mid], int(cfg[\'rescue\']))\n            assert 1 <= len(new) <= 25 and len(set(new)) == len(new), mid\n            if not rescued:\n                assert sorted(new) == sorted(original), f\'candidate set changed: {mid}\'\n            stats[\'active_rows\'] += int(active)\n            stats[\'changed_rows\'] += int(new != original)\n            stats[\'changed_top1\'] += int(new[0] != original[0])\n            stats[\'rescued\'] += int(rescued)\n            rows.append((mid, \';\'.join(new)))\n            if active:\n                b_rank = {s: i + 1 for i, s in enumerate(original)}\n                n_rank = dict(zip(g.smiles, stable_rank(g.neural_score.values)))\n                for pos, s in enumerate(new, 1):\n                    moves.append((name, mid, s, pos, b_rank.get(s, 0), int(n_rank.get(s, 0))))\n        out = pd.DataFrame(rows, columns=[\'molecule_id\', \'smiles\'])\n        assert out.molecule_id.tolist() == base.molecule_id.tolist()\n        path = outdir / f\'submission_{name}.csv\'\n        out.to_csv(path, index=False)\n        report[name] = dict(cfg, offset=offset, file=path.name, sha256=sha256(path), **stats)\n    moves = pd.DataFrame(moves, columns=[\'variant\', \'molecule_id\', \'smiles\', \'final_rank\', \'baseline_rank\', \'neural_rank\'])\n    (outdir / \'consensus_report.json\').write_text(json.dumps(report, indent=2))\n    return report, moves\n\n\ndef validate(path, sample_ids, baseline_path=None, allow_new=False, identity_of=None):\n    """Raise unless `path` is a well-formed answer file for the expected molecules."""\n    frame = pd.read_csv(path, dtype=str, keep_default_na=False)\n    assert list(frame.columns) == [\'molecule_id\', \'smiles\'], \'columns\'\n    assert frame.molecule_id.tolist() == list(sample_ids), \'molecule order\'\n    lists = frame.smiles.str.split(\';\')\n    assert lists.map(len).between(1, 25).all(), \'list length\'\n    assert lists.map(lambda v: all(x.strip() for x in v) and len(set(v)) == len(v)).all(), \'empty or repeated SMILES\'\n    if baseline_path is not None:\n        base = pd.read_csv(baseline_path, dtype=str, keep_default_na=False)\n        assert base.molecule_id.tolist() == frame.molecule_id.tolist(), \'baseline order\'\n        for before, after in zip(base.smiles, frame.smiles):\n            b, a = before.split(\';\'), after.split(\';\')\n            if allow_new:\n                assert len(a) >= min(len(b), 25), \'list got shorter\'\n            else:\n                assert sorted(b) == sorted(a), \'candidate set changed\'\n    if identity_of is not None:\n        for answer in frame.smiles:\n            keys = [identity_of.get(s) for s in answer.split(\';\')]\n            keys = [k for k in keys if k]\n            assert len(keys) == len(set(keys)), \'duplicate metric identity\'\n    return frame\n'
MODULES['outsiders.py'] = '"""Collect candidates that the frozen pipeline considered but left outside its final list.\n\nReads the pipeline\'s own record of its candidate lists (e004_candidate_inputs.json),\nwalks the fusion overflow, Engine 2, the main-engine tail and the PubChem channel in\nround-robin rank order, and keeps structures whose official metric identity (the frozen\nnotebook\'s own metric_identity, executed from its source) is new for the molecule.\nOutput rows follow the evidence schema the neural scorer reads; baseline_rank 26, 27, ...\nonly fixes their order.\n\nusage: python outsiders.py <frozen.ipynb> <root> <evidence.parquet> <out.parquet> [per_molecule] [max_checks]\n"""\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nimport pandas as pd\n\nCOLUMNS = [\'molecule_id\', \'smiles\', \'metric_key\', \'source\', \'source_rank\', \'baseline_rank\', \'processed\']\n\n\ndef load_identity(frozen):\n    cells = json.loads(Path(frozen).read_text())[\'cells\']\n    hits = [\'\'.join(c[\'source\']) for c in cells\n            if c[\'cell_type\'] == \'code\' and \'def metric_identity\' in \'\'.join(c[\'source\'])]\n    assert len(hits) == 1, \'metric identity cell not found in the frozen source\'\n    scope = {\'__name__\': \'frozen_metric_identity\'}\n    exec(compile(hits[0], \'frozen_metric_identity\', \'exec\'), scope)\n    return scope[\'metric_identity\']\n\n\ndef source_lists(payload, mid):\n    eng = payload.get(\'engine2\', {}).get(mid) or {}\n    pc = payload.get(\'pubchem\', {}).get(mid) or {}\n    return [(\'fusion_overflow\', list(payload.get(\'fusion_overflow\', {}).get(mid) or [])),\n            (\'engine2\', list(eng.get(\'smiles\') or [])),\n            (\'main_tail\', list(payload.get(\'base_tail\', {}).get(mid) or [])),\n            (\'pubchem\', list(pc.get(\'pc\') or []))]\n\n\ndef collect(payload, evidence, identity, per_molecule=12, max_checks=30):\n    rows = []\n    active = evidence[evidence.processed.astype(bool)]\n    for mid, g in active.groupby(\'molecule_id\', sort=False):\n        mid = str(mid)\n        if mid not in payload.get(\'prefusion\', {}):\n            continue\n        listed = g.sort_values(\'baseline_rank\', kind=\'stable\').smiles.astype(str).tolist()\n        taken = {k for k in g.metric_key.tolist() if isinstance(k, str) and k}\n        seen = set(listed)\n        lists = source_lists(payload, mid)\n        depth = max((len(v) for _, v in lists), default=0)\n        picked = checks = 0\n        for r in range(depth):\n            if picked >= per_molecule or checks >= max_checks:\n                break\n            for name, values in lists:\n                if r >= len(values) or picked >= per_molecule or checks >= max_checks:\n                    continue\n                s = str(values[r]).strip()\n                if not s or s == \'CCO\' or s in seen:\n                    continue\n                seen.add(s)\n                checks += 1\n                key = identity(s)\n                if key is None or key[0] in taken:\n                    continue\n                taken.add(key[0])\n                picked += 1\n                rows.append((mid, s, key[0], name, r + 1, 25 + picked, True))\n    return pd.DataFrame(rows, columns=COLUMNS)\n\n\nif __name__ == \'__main__\':\n    frozen, root, evidence_path, out = sys.argv[1:5]\n    per = int(sys.argv[5]) if len(sys.argv) > 5 else 12\n    checks = int(sys.argv[6]) if len(sys.argv) > 6 else 30\n    t0 = time.time()\n    identity = load_identity(frozen)\n    payload = json.loads((Path(root) / \'e004_candidate_inputs.json\').read_text())\n    evidence = pd.read_parquet(evidence_path)\n    frame = collect(payload, evidence, identity, per, checks)\n    frame.to_parquet(out, index=False)\n    by_source = frame.source.value_counts().to_dict() if len(frame) else {}\n    print(\'OUTSIDERS\', len(frame), \'candidates for\', frame.molecule_id.nunique(), \'molecules\',\n          json.dumps(by_source), f\'{time.time() - t0:.0f}s\', flush=True)\n'
MODULE_SHA256 = {
 "runner.py": "86b74f3b4cd5767a2cbe1d291735c292b66b4fbf286a705c6cb777a1865883db",
 "export_evidence.py": "d4ec36a9f6216d2607fff4f481bc465dbc8fbc616ec4b24f12932e4c5e546e19",
 "neural_scores.py": "cb9e4d6732992e3310e951533fd0c644d3b515134ad03768c35e5a86551267d2",
 "submission.py": "d0027a893b1bfefe8066ede4f625c584fe0d4fa16bfe4bee32265ef34a7f007c",
 "ranking.py": "6f909cd7e41e88387c23109b93dd590885d2283509804279a58c624a758987d7",
 "bundle.py": "8595ef16ead9c9e6ac34a3398bff4b748f5668cdc517c396be18100a5b745cea",
 "consensus.py": "9e67ab49a852404587e62ca84e0fc551d70ce68ec8f7a47739aada6ca08d3989",
 "outsiders.py": "4dcf6c2b9c9b7be3ac8f7a616683495430fcb4e0c25b449684d132daaa8e7b5c"
}
MODEL_PINS = {
 "features.py": "a97ad9b3ab3711517864d56e497e1672cb29bb05683c72b52db9f9b0a17b1455",
 "model.py": "2547418857386d1c2513fd1ab6c70961e6a96e73d9d495bbfe3052197e69dc39",
 "config.json": "563fb2a2a81ca48ea094f9ab78c157a48a6e9a5503e85e6ce8f0fb3615c1cc03",
 "final_model.pt": "44436b2daec64373e66f9120dd2389c24fc83dd9430ebeeac18e3cb49dcb9642"
}
FROZEN_SHA256 = 'c0fa8885956fe1413707df55bed596c6e42f7de134be8d2042fc28e80f3566c3'
FROZEN_B64 = '...'
print(f'{len(MODULES)} modules and the pinned pipeline source ready ({len(FROZEN_B64) * 3 // 4 / 1e6:.2f} MB)')

In [ ]:
# ── Stage machinery: process runner, compact trace, and the four stage functions ───────────────
import contextlib, importlib

STAGES = []           # (stage, seconds, status) for the run summary
STAGE_MARKS = {}      # stage -> [(frozen cell, START/DONE, seconds since stage start)]
FROZEN_STAGE = {
    2: 'Official RDKit 2026.03.3', 6: 'Competition files', 9: 'Asset inventory', 12: 'Documentation prior table',
    14: 'Prior benchmark table', 16: 'Natural-product panel', 18: 'Natural-product knowledge table',
    26: 'Datasets, priors and wheels', 27: 'Determinism patch', 28: 'Preflight', 30: 'Commit or rerun switch',
    32: 'Engine 2 · two rankers', 34: 'PubChem channel', 36: 'Engine 1 · library, pool, spectrum cache',
    37: 'Engine 1 · models and rankers', 39: 'Engine 1 · candidate lists', 41: 'GLACIER + ICEBERG forward models',
    43: 'Gated PubChem merge', 45: 'Two-engine fusion', 48: 'Metric audit and safe tail', 49: 'Metric-identity fusion',
    50: 'Postflight record', 51: 'Fragment score gap', 52: 'Window-40 overlay', 53: 'Engine 2 rank-2 placement',
    54: 'Execution receipt', 56: 'Example answer table'}
CELL_MARK = re.compile(r'^PIPELINE CELL (\d+) (START|DONE)')
KEY_LINE = re.compile(r'FAILED|ERROR|Traceback|merge stats|RANK2_OVERLAY|EXPORTED FROZEN|NEURAL FEATURES|'
                      r'NEURAL input failures|OUTSIDERS|engine lists \d+ |pubchem lists \d+|errors \d+ \|')
BASELINE = ROOT / 'submission_baseline0465.csv'
VARIANTS_DIR = ROOT / 'variants'
RESCUE_SLOTS = {'V3': 3, 'CUSTOM': int(CUSTOM.get('rescue', 0))}.get(VARIANT, 0)
PIPELINE_DONE = NEURAL_OK = OUTSIDERS_OK = CONSENSUS_OK = False
REPORT, MOVES, SUBMITTED, consensus = {}, pd.DataFrame(), None, None


def run_stage(args, log, name, quiet=None):
    """Run one stage in its own process; keep the full log on disk and print a compact trace."""
    quiet = QUIET_LOGS if quiet is None else quiet
    env = {**os.environ, 'PYTHONHASHSEED': '0', 'PYTHONUNBUFFERED': '1'}
    env.pop('INC_PANEL', None)
    t0, marks, tail = time.time(), [], collections.deque(maxlen=150)
    print(f'▶ {name}', flush=True)
    with open(log, 'w') as handle:
        proc = subprocess.Popen([str(a) for a in args], cwd=ROOT, env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            handle.write(line); handle.flush()
            t = time.time() - t0
            tail.append(line)
            m = CELL_MARK.match(line)
            if m:
                cell, kind = int(m.group(1)), m.group(2)
                marks.append((cell, kind, t))
                if quiet and kind == 'DONE' and cell in FROZEN_STAGE:
                    start = next((s for c, k, s in reversed(marks) if c == cell and k == 'START'), t)
                    if t - start >= 1:
                        print(f'  ✓ {t / 60:6.1f} min   {FROZEN_STAGE[cell]:<40s} {t - start:8,.0f} s', flush=True)
                continue
            if not quiet:
                if len(line) < 5000:
                    print(line, end='', flush=True)
            elif KEY_LINE.search(line) and len(line) < 800:
                print('      ' + line.strip()[:180], flush=True)
        status = proc.wait()
    seconds = time.time() - t0
    STAGE_MARKS[name] = marks
    STAGES.append((name, seconds, 'ok' if status == 0 else f'exit {status}'))
    if status:
        print(''.join(tail))
        raise RuntimeError(f'{name} failed with exit code {status}; full log: {log}')
    print(f'  done in {minutes(seconds)}', flush=True)
    return seconds


def materialise_sources():
    """Stage 0: write and verify the pinned sources, find the neural bundle, install the official RDKit."""
    global FROZEN
    t0 = time.time()
    for name, text in MODULES.items():
        (CODE / name).write_text(text)
        assert hashlib.sha256((CODE / name).read_bytes()).hexdigest() == MODULE_SHA256[name], name
    FROZEN = CODE / 'frozen_0465.ipynb'
    FROZEN.write_bytes(base64.b64decode(FROZEN_B64))
    assert hashlib.sha256(FROZEN.read_bytes()).hexdigest() == FROZEN_SHA256, 'Pinned pipeline source differs'
    assert sys.version_info[:2] == (3, 12), 'The pinned pipeline needs the Python 3.12 Kaggle image (pin the environment)'
    if str(CODE) not in sys.path:
        sys.path.insert(0, str(CODE))
    from bundle import find_bundle
    bundle = find_bundle(str(INPUT), ROOT, MODEL_PINS)
    (CODE / 'bootstrap.py').write_text(''.join(json.loads(FROZEN.read_text())['cells'][2]['source']))
    run_stage([sys.executable, CODE / 'bootstrap.py'], ROOT / 'bootstrap.log', 'Official RDKit wheel', quiet=False)
    STAGES[-1] = ('Sources and RDKit wheel', time.time() - t0, 'ok')
    tiles([('Pipeline source', FROZEN_SHA256[:12] + '…', 'SHA-256 verified'),
           ('Helper modules', f'{len(MODULES)}', '6 public + 2 new, SHA-256 verified'),
           ('Neural bundle', Path(bundle).name, '4 files, SHA-256 verified'),
           ('Stage 0', minutes(time.time() - t0), 'including the wheel install')])
    return bundle


def run_public_pipeline():
    """Stage 1: every cell of the pinned pipeline in one process, then the export of its final lists."""
    if not RUN_PIPELINE:
        note('Fast commit.', 'The pipeline is skipped in this commit run; the scoring rerun executes every stage.', BLUE)
        return False
    assert not (ROOT / 'work/spec_meta.parquet').exists() and not (ROOT / 'submission.csv').exists(), \
        'Start from a fresh session (Run All after a restart)'
    seconds = run_stage([sys.executable, CODE / 'runner.py', FROZEN, FROZEN_SHA256, ROOT],
                        ROOT / 'submission_pipeline.log', 'Public 0.465 pipeline')
    receipt = json.loads((ROOT / 'execution_receipt.json').read_text())
    full = receipt['is_full_run']
    tiles([('Molecules ranked', f"{receipt['processed_molecules']:,}", 'hidden test' if full else 'visible smoke subset'),
           ('Answer rows', f'{len(SAMPLE_IDS):,}', 'all ranked' if full else 'placeholders elsewhere'),
           ('Pipeline time', minutes(seconds), 'one process, every cell'),
           ('Baseline file', BASELINE.name, 'kept for reference')])
    return True


def score_final_lists():
    """Stage 2a: the public scorer on exactly the exported candidates."""
    try:
        run_stage([sys.executable, CODE / 'neural_scores.py', BUNDLE, 'final_model.pt',
                   ROOT / 'frozen_candidate_evidence.parquet', ROOT / 'effective_query_spectra.parquet',
                   ROOT / 'neural_candidate_features.parquet', 'cuda'],
                  ROOT / 'submission_neural.log', 'Neural scorer · final lists')
        return True
    except Exception as exc:
        note('Neural scorer failed.', f'The 0.465 answers will be submitted. {type(exc).__name__}: {str(exc)[:300]}', RED)
        return False


def score_outside_candidates(needed):
    """Stage 2b (optional): collect and score candidates ranked just outside the final lists."""
    elapsed_h = (time.time() - NB_T0) / 3600
    if not needed:
        return False
    if elapsed_h >= TIME_GUARD_HOURS:
        note('Tail rescue skipped.', f'The session has run {elapsed_h:.1f} h (guard {TIME_GUARD_HOURS} h).', YELLOW)
        return False
    try:
        run_stage([sys.executable, CODE / 'outsiders.py', FROZEN, ROOT, ROOT / 'frozen_candidate_evidence.parquet',
                   ROOT / 'outsider_evidence.parquet', OUTSIDERS_PER_MOLECULE],
                  ROOT / 'outsiders.log', 'Outside candidates · collect')
        if not len(pd.read_parquet(ROOT / 'outsider_evidence.parquet')):
            return False
        run_stage([sys.executable, CODE / 'neural_scores.py', BUNDLE, 'final_model.pt',
                   ROOT / 'outsider_evidence.parquet', ROOT / 'effective_query_spectra.parquet',
                   ROOT / 'outsider_neural_features.parquet', 'cuda'],
                  ROOT / 'outsiders_neural.log', 'Neural scorer · outside candidates')
        return True
    except Exception as exc:
        note('Tail rescue disabled.', f'{type(exc).__name__}: {str(exc)[:300]}', YELLOW)
        return False


def show_neural_summary():
    if not NEURAL_OK:
        return
    nf = pd.read_parquet(ROOT / 'neural_candidate_features.parquet')
    act = nf[nf.processed.astype(bool)]
    n_out = len(pd.read_parquet(ROOT / 'outsider_neural_features.parquet')) if OUTSIDERS_OK else 0
    tiles([('Candidates scored', f'{len(act):,}', f'{act.molecule_id.nunique():,} molecules'),
           ('Scored molecules', f'{int(act.groupby("molecule_id").neural_ok.all().sum()):,}', 'all spectra usable'),
           ('Spectra per molecule', f'{act.groupby("molecule_id").n_spectra.first().median():.0f}', 'median used by the scorer'),
           ('Outside candidates', f'{n_out:,}', 'scored for the tail rescue' if n_out else 'not needed for this variant')])


def build_consensus():
    """Stage 3a: the public 0.466 file, its exact rebuild, then every variant."""
    global consensus
    try:
        consensus = importlib.import_module('consensus')
        from submission import apply as public_apply
        evidence = ROOT / 'neural_candidate_features.parquet'
        with contextlib.redirect_stdout(io.StringIO()):      # the public helper prints its own report
            public_apply(evidence, BASELINE, ROOT / 'submission_public0466.csv', weight=0.25, offset=3.0)
        consensus.build_variants(evidence, BASELINE, ROOT / 'variants_check', {'V1': consensus.PRESETS['V1']}, 3.0)
        mine = pd.read_csv(ROOT / 'variants_check/submission_V1.csv', dtype=str, keep_default_na=False)
        public = pd.read_csv(ROOT / 'submission_public0466.csv', dtype=str, keep_default_na=False)
        assert mine.equals(public), 'V1 differs from the public 0.466 file'
        presets = dict(consensus.PRESETS)
        if VARIANT == 'CUSTOM':
            presets['CUSTOM'] = dict(w_top=float(CUSTOM['w_top']), w_mid=float(CUSTOM['w_mid']),
                                     rescue=int(CUSTOM.get('rescue', 0)), label='Custom weights')
        report, moves = consensus.build_variants(
            evidence, BASELINE, VARIANTS_DIR, presets, RRF_OFFSET,
            ROOT / 'outsider_neural_features.parquet' if OUTSIDERS_OK else None)
        print('V1 rebuilt exactly: the new code reproduces the public 0.466 file row for row.')
        return True, report, moves
    except Exception as exc:
        note('Consensus layer disabled.', f'{type(exc).__name__}: {str(exc)[:400]}', RED)
        return False, {}, pd.DataFrame()


def identity_map():
    keys = {}
    for path in (ROOT / 'neural_candidate_features.parquet', ROOT / 'outsider_neural_features.parquet'):
        if path.is_file():
            f = pd.read_parquet(path, columns=['smiles', 'metric_key'])
            keys.update({s: k for s, k in zip(f.smiles, f.metric_key) if isinstance(k, str)})
    return keys


def well_formed(path, same_candidates=True):
    """Minimal checks that do not depend on the new modules."""
    f = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert list(f.columns) == ['molecule_id', 'smiles'] and f.molecule_id.tolist() == SAMPLE_IDS, 'rows'
    lists = f.smiles.str.split(';')
    assert lists.map(len).between(1, 25).all(), 'list length'
    assert lists.map(lambda v: all(x.strip() for x in v) and len(set(v)) == len(v)).all(), 'SMILES'
    if same_candidates and BASELINE.is_file():
        b = pd.read_csv(BASELINE, dtype=str, keep_default_na=False)
        assert all(sorted(x.split(';')) == sorted(y.split(';')) for x, y in zip(b.smiles, f.smiles)), 'candidates'


def choose_and_submit():
    """Stage 3b: chosen variant -> V1 -> V0 -> public 0.466 file -> baseline file (-> sample in a fast commit)."""
    t0 = time.time()
    chain = []
    if CONSENSUS_OK:
        chain += [(n, VARIANTS_DIR / f'submission_{n}.csv') for n in dict.fromkeys([VARIANT, 'V1', 'V0'])]
    chain += [('public 0.466 file', ROOT / 'submission_public0466.csv'), ('baseline file', BASELINE)]
    chosen = None
    for name, path in chain:
        if not path.is_file():
            continue
        try:
            rescue = name in REPORT and REPORT[name].get('rescued', 0) > 0
            if rescue:
                consensus.validate(path, SAMPLE_IDS, BASELINE, allow_new=True, identity_of=identity_map())
            well_formed(path, same_candidates=not rescue)
            chosen = (name, path)
            break
        except Exception as exc:
            note(f'{name} rejected.', f'{type(exc).__name__}: {str(exc)[:300]}', YELLOW)
    if chosen is None:
        assert not IS_RERUN, 'No valid answer file in the scoring rerun'
        chosen = ('sample (fast commit)', COMP / 'sample_submission.csv')
    shutil.copyfile(chosen[1], ROOT / 'submission.csv')
    STAGES.append(('Consensus and checks', time.time() - t0, 'ok'))
    return chosen[0]


def show_variant_board():
    if CONSENSUS_OK:
        board = pd.DataFrame([{
            'Variant': k, 'Rule': v['label'], 'Leader weight': f"{v['w_top']:.2f}",
            'Ranks 2–25 weight': f"{v['w_mid']:.2f}", 'Tail rescue': v['rescue'] or '–',
            'Rows re-ordered': f"{v['changed_rows']:,}", 'Rank 1 changed': f"{v['changed_top1']:,}",
            'Rescued': f"{v['rescued']:,}"} for k, v in REPORT.items()])
        board.index = board.Variant
        table_html(board, highlight=SUBMITTED, numeric=board.columns[2:])
    sub = pd.read_csv(ROOT / 'submission.csv', dtype=str, keep_default_na=False)
    tiles([('Submitted', SUBMITTED, 'copied to submission.csv'),
           ('Rows', f'{len(sub):,}', 'one per molecule, sample order'),
           ('Candidates per row', f'{sub.smiles.str.count(";").add(1).median():.0f}', 'median, at most 25'),
           ('SHA-256', hashlib.sha256((ROOT / 'submission.csv').read_bytes()).hexdigest()[:12] + '…', 'of submission.csv')])

In [ ]:
# Stage 0 · write the pinned sources, verify every SHA-256, find the neural bundle, install the official RDKit wheel
BUNDLE = materialise_sources()

### Stage 1 · the public 0.465 pipeline

`runner.py` from the 0.466 notebooks executes every code cell of the pinned pipeline in one process, exactly as published, then exports the final 25 candidates of every molecule with the evidence each engine gave them (`frozen_candidate_evidence.parquet`). The trace below prints one line per pipeline stage with its end time and duration.

In [ ]:
# Stage 1 · the public 0.465 pipeline: every cell of the pinned notebook, then the export of its final lists
PIPELINE_DONE = run_public_pipeline()

### Stage 2 · neural spectrum&nbsp;&harr;&nbsp;structure scores

The scorer of the 0.466 notebooks embeds every query spectrum of a molecule and every candidate structure in a shared space, adds a predicted-fingerprint match, and averages over the molecule's spectra. It runs after the pipeline process has exited, so it has the GPU to itself, and it scores exactly the candidates of the exported lists. When the chosen variant uses the tail rescue (and in the commit run, for the figures), the same scorer also rates up to 12 candidates per molecule that the pipeline had ranked just outside its list: the fusion overflow, Engine 2, the Engine 1 tail and the PubChem channel, de-duplicated with the pipeline's copy of the official metric identity.

In [ ]:
# Stage 2 · neural scores: always for the final lists; for outside candidates only when the tail rescue needs them
NEURAL_OK = PIPELINE_DONE and score_final_lists()
OUTSIDERS_OK = NEURAL_OK and score_outside_candidates(needed=RESCUE_SLOTS > 0 or SHOW)
show_neural_summary()

### Stage 3 · leader-safe consensus

For a molecule with baseline ranks $b_i$ and neural ranks $n_i$ over the same candidates (offset $k = 3$):

$$\text{rank 1} = \arg\max_i \left[\frac{1}{k+b_i} + \frac{w_\text{top}}{k+n_i}\right], \qquad \text{ranks } 2\text{–}25:\ \text{the rest by } \frac{1}{k+b_i} + \frac{w_\text{mid}}{k+n_i}.$$

With $w_\text{top} = w_\text{mid} = 0.25$ this is the public 0.466 recipe (V1). V2 keeps $w_\text{top} = 0.25$, so its leader is always V1's leader, and sets $w_\text{mid} = 0.5$. At that weight, a candidate in third place that the neural model ranks first moves to second unless the current second is also in the neural top two; at the top, the public rule still needs a rank-2 candidate to be the neural favourite and the leader to fall below neural rank 17 before rank 1 changes.

The stage then runs three checks before anything is submitted:

1. **Reproduction.** V1 is rebuilt by this notebook's code and compared row by row with the file written by the public `submission.apply`; any difference disables every new variant.
2. **Validity.** The chosen file must list the sample molecules in order, with 1 to 25 distinct non-empty SMILES each, and keep each molecule's candidate set (the tail rescue may only swap the last three places for new metric identities).
3. **Fallback.** If a check fails, the next file in the chain is tried: chosen variant → V1 → V0 → the public 0.466 file → the 0.465 answers, which always exist once the pipeline has run.

In [ ]:
# Stage 3 · rebuild the public 0.466 file exactly, build every variant, then choose and validate the answer file
if NEURAL_OK:
    CONSENSUS_OK, REPORT, MOVES = build_consensus()
SUBMITTED = choose_and_submit()
show_variant_board()

<a id="changed"></a>
## 5 · What changed

The figures below are drawn in the commit run only. Its 12 molecules come from the training library, where the pipeline usually has an exact library match and the neural model tends to agree with it; on the hidden test the two disagree more often, and the layer earns its keep there. Read these panels as a check that the layer does what it says: it keeps the public rule's leader, moves candidates within the list, and never adds a structure the pipeline did not propose.

In [ ]:
# ── Figures 4–6 · run timeline, re-ranking diagnostics and one molecule's answer ────────────
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

SEQ_BLUE = LinearSegmentedColormap.from_list('seq_blue', ['#cde2fb', '#86b6ef', '#3987e5', '#256abf', '#104281', '#0d366b'])
NEW_STAGES = {'Outside candidates · collect', 'Neural scorer · outside candidates', 'Consensus and checks'}
PUBLIC_SCORER = {'Neural scorer · final lists'}


def run_timeline():
    rows, t, small = [], 0.0, 0.0
    for name, seconds, _ in STAGES:
        marks = STAGE_MARKS.get(name, [])
        if name == 'Public 0.465 pipeline' and marks:
            starts = {}
            for cell, kind, ts in marks:
                if kind == 'START':
                    starts[cell] = ts
                elif cell in starts:
                    d = ts - starts[cell]
                    if d >= 20:
                        rows.append((FROZEN_STAGE.get(cell, f'pipeline cell {cell}'), t + starts[cell], d, 'pipeline'))
                    else:
                        small += d
        else:
            kind = 'new' if name in NEW_STAGES else ('scorer' if name in PUBLIC_SCORER else 'pipeline')
            rows.append((name, t, seconds, kind))
        t += seconds
    colors = {'pipeline': BLUE, 'scorer': AQUA, 'new': VIOLET}
    fig, ax = plt.subplots(figsize=(13, 0.36 * len(rows) + 1.4))
    for i, (lab, start, dur, kind) in enumerate(rows):
        ax.barh(i, dur / 60, left=start / 60, height=0.56, color=colors[kind])
        ax.text((start + dur) / 60 + t / 60 * 0.006, i, minutes(dur), va='center', fontsize=9, color=INK2)
    ax.set_yticks(range(len(rows)), [r[0] for r in rows]); ax.invert_yaxis()
    ax.tick_params(axis='y', length=0); ax.grid(axis='y', visible=False)
    ax.set_xlim(0, t / 60 * 1.1); ax.set_xlabel('minutes since the notebook started its first stage')
    ax.set_title(f'Run timeline · {minutes(t)} in stages' + (f' (cells under 20 s: {minutes(small)} in total)' if small else ''))
    ax.legend(handles=[Patch(color=BLUE, label='public pipeline'), Patch(color=AQUA, label='public neural scorer'),
                       Patch(color=VIOLET, label='this notebook')], loc='upper right')
    plt.show()


def pick_example(moves, variant):
    m = moves[(moves.variant == variant) & (moves.baseline_rank > 0)]
    if m.empty:
        return None
    shift = (m.final_rank - m.baseline_rank).abs().groupby(m.molecule_id, sort=False).sum()
    return shift.sort_values(ascending=False, kind='stable').index[0]


def rank_flow(ax, moves, variant, mid):
    m = moves[(moves.variant == variant) & (moves.molecule_id == mid)]
    outside = 27
    for r in m.itertuples(index=False):
        b = r.baseline_rank if r.baseline_rank > 0 else outside
        up = b - r.final_rank
        if r.baseline_rank == 0:
            col, lw, z = AQUA, 2.0, 4
        elif up >= 2:
            col, lw, z = VIOLET, 2.0, 3
        elif up <= -2:
            col, lw, z = QUIET_GRAY, 1.3, 2
        else:
            col, lw, z = '#e3e1da', 1.1, 1
        ax.plot([0, 1], [b, r.final_rank], color=col, lw=lw, solid_capstyle='round', zorder=z)
        ax.plot([0, 1], [b, r.final_rank], 'o', color=col, ms=6, mec=SURFACE, mew=1.5, zorder=z + 1)
    ax.set_xlim(-0.25, 1.25); ax.set_ylim(outside + 1 if (m.baseline_rank == 0).any() else 26, 0)
    ax.set_xticks([0, 1], ['baseline 0.465', variant]); ax.grid(False)
    ticks = [1, 5, 10, 15, 20, 25] + ([outside] if (m.baseline_rank == 0).any() else [])
    ax.set_yticks(ticks, [str(x) if x != outside else 'outside' for x in ticks])
    ax.spines['left'].set_visible(False); ax.spines['bottom'].set_visible(False); ax.tick_params(length=0)
    ax.set_ylabel('rank')
    ax.set_title(f'One list re-ordered · {mid}')
    ax.legend(handles=[Line2D([], [], color=VIOLET, lw=2, label='moved up 2+'),
                       Line2D([], [], color=QUIET_GRAY, lw=1.3, label='moved down 2+'),
                       Line2D([], [], color='#e3e1da', lw=1.1, label='within one place')] +
                      ([Line2D([], [], color=AQUA, lw=2, label='rescued from outside')] if (m.baseline_rank == 0).any() else []),
              loc='lower center', bbox_to_anchor=(0.5, -0.3), ncol=2, fontsize=9)


def rank_heatmap(ax, moves, variant):
    m = moves[(moves.variant == variant) & (moves.baseline_rank > 0)]
    grid = np.zeros((25, 25))
    np.add.at(grid, (m.baseline_rank.values - 1, m.final_rank.values - 1), 1)
    shown = np.ma.masked_where(grid == 0, grid)
    im = ax.imshow(shown, cmap=SEQ_BLUE, origin='upper', extent=(0.5, 25.5, 25.5, 0.5), interpolation='nearest')
    ax.plot([0.5, 25.5], [0.5, 25.5], color=AXIS, lw=0.8)
    ax.set_xticks([1, 5, 10, 15, 20, 25]); ax.set_yticks([1, 5, 10, 15, 20, 25]); ax.grid(False)
    ax.set_xlabel(f'rank in {variant}'); ax.set_ylabel('baseline rank')
    ax.set_title(f'All ranked molecules · {m.molecule_id.nunique()} lists')
    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.03); cb.outline.set_visible(False)
    cb.set_label('candidates', color=INK2)


def mol_png(smiles, w=420, h=320):
    from rdkit import Chem
    from rdkit.Chem.Draw import rdMolDraw2D
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    d = rdMolDraw2D.MolDraw2DCairo(w, h)
    o = d.drawOptions(); o.setBackgroundColour((0.988, 0.988, 0.984, 1.0)); o.bondLineWidth = 2; o.padding = 0.06
    rdMolDraw2D.PrepareAndDrawMolecule(d, mol); d.FinishDrawing()
    return plt.imread(io.BytesIO(d.GetDrawingText()), format='png')


def molecule_card(variant, mid):
    answer = pd.read_csv(VARIANTS_DIR / f'submission_{variant}.csv', dtype=str).set_index('molecule_id').smiles[mid].split(';')
    ev = pd.read_parquet(ROOT / 'neural_candidate_features.parquet')
    ev = ev[ev.molecule_id == mid].set_index('smiles')
    n_rank = dict(zip(ev.index, ev.neural_score.rank(ascending=False, method='first').astype(int)))
    q = pd.read_parquet(ROOT / 'effective_query_spectra.parquet')
    q = q[q.molecule_id.astype(str) == mid]
    fig = plt.figure(figsize=(14.5, 3.9))
    gs = fig.add_gridspec(1, 4, width_ratios=[1.45, 1, 1, 1], wspace=0.08)
    ax = fig.add_subplot(gs[0, 0])
    if len(q):
        r = q.loc[q.ms2_mzs.map(len).idxmax()]
        mz = np.asarray(r.ms2_mzs, float); it = np.asarray(r.ms2_normalized_intensities, float)
        it = it / it.max() if it.size and it.max() > 0 else it
        ax.vlines(mz, 0, it, color=BLUE, lw=1.1)
        ax.plot([r.precursor_mz], [1.06], 'v', color=ORANGE, ms=6)
        ax.set_ylim(0, 1.2); ax.set_xlim(0, float(r.precursor_mz) * 1.08); ax.grid(axis='x', visible=False)
        ax.set_title(f'{mid} · {r.adduct} · {len(mz)} peaks', fontsize=11.5)
        ax.set_xlabel('m/z'); ax.set_ylabel('relative intensity')
    for j, smi in enumerate(answer[:3]):
        axm = fig.add_subplot(gs[0, j + 1]); axm.axis('off')
        try:
            img = mol_png(smi)
        except Exception:
            img = None
        if img is not None:
            axm.imshow(img)
        else:
            axm.text(0.5, 0.5, smi[:60], ha='center', va='center', fontsize=8, color=INK2, wrap=True)
        if smi in ev.index:
            e = ev.loc[smi]
            src = [n for n, c in (('main', 'main_rank'), ('engine 2', 'engine2_rank'), ('PubChem', 'pubchem_rank'))
                   if float(e.get(c, 0) or 0) > 0]
            sub = f"baseline #{int(e.baseline_rank)} · neural #{n_rank.get(smi, 0)}\n{' · '.join(src) or 'tail fill'}"
        else:
            sub = 'rescued from outside the list'
        axm.set_title(f'#{j + 1} in {variant}', fontsize=12, loc='center')
        axm.text(0.5, -0.04, sub, transform=axm.transAxes, ha='center', va='top', fontsize=9.5, color=INK2)
    plt.show()


if SHOW and CONSENSUS_OK:
    try:
        run_timeline()
    except Exception as exc:
        print('timeline skipped:', type(exc).__name__, exc)
    focus = SUBMITTED if SUBMITTED in REPORT and SUBMITTED not in ('V0', 'V1') else 'V2'
    base_moves = MOVES[MOVES.variant == 'V0']
    rho = base_moves.groupby('molecule_id').apply(
        lambda g: np.corrcoef(g.baseline_rank, g.neural_rank)[0, 1] if len(g) > 2 else np.nan).dropna()
    fm = MOVES[(MOVES.variant == focus) & (MOVES.baseline_rank > 0)]
    tiles([('Variant shown', focus, REPORT[focus]['label']),
           ('Rows re-ordered', f"{REPORT[focus]['changed_rows']:,}", f"of {REPORT[focus]['active_rows']:,} ranked"),
           ('Rank 1 changed', f"{REPORT[focus]['changed_top1']:,}", 'vs the 0.465 answers'),
           ('Mean shift', f'{(fm.final_rank - fm.baseline_rank).abs().mean():.2f}', 'places per candidate'),
           ('Agreement', f'{rho.median():.2f}' if len(rho) else 'n/a', 'median rank correlation, pipeline vs neural')])
    example = pick_example(MOVES, focus)
    if example is not None:
        try:
            fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5.6), gridspec_kw=dict(width_ratios=[1, 1.2], wspace=0.3))
            rank_flow(a1, MOVES, focus, example)
            rank_heatmap(a2, MOVES, focus)
            plt.show()
        except Exception as exc:
            print('re-ranking figure skipped:', type(exc).__name__, exc)
        try:
            molecule_card(focus, example)
        except Exception as exc:
            print('molecule card skipped:', type(exc).__name__, exc)
elif SHOW:
    note('No diagnostics.', 'The consensus layer did not run in this session (see the stage notes above).', BLUE)

<a id="plan"></a>
## 6 · Variants and submission plan

| Variant | Rank 1 | Ranks 2–25 | Tail | Use it for |
|---|---|---|---|---|
| **V0** | pipeline | pipeline | — | the 0.465 control |
| **V1** | neural RRF 0.25 | neural RRF 0.25 | — | the public 0.466 recipe, rebuilt exactly |
| **V2** · default | as V1 | neural RRF **0.50** | — | the leader-safe consensus |
| **V3** | as V1 | neural RRF 0.50 | 3-slot rescue | V2 plus coverage for answers ranked just outside the list |
| **V4** | as V1 | neural RRF **1.00** | — | a high-variance probe of how much the neural model knows |

All five files are written on every run (`/kaggle/working/variants/`), but only `submission.csv` is scored, so each variant needs its own submission. A sensible order:

1. **V2.** Its leader is V1's leader, so any difference from the public 0.466 comes from ranks 2–25.
2. If V2 scores above 0.466, **V4** tells you whether the neural model deserves even more weight in the middle of the list.
3. If V2 does not, the neural model sits below the toy model's crossover on the hidden spectra: try `CUSTOM` with `w_mid=0.35`, or keep V1's order and add only the tail rescue (`CUSTOM = dict(w_top=0.25, w_mid=0.25, rescue=3)`).
4. **V3** adds the tail rescue to V2; it is worth a submission once the ordering question is settled.

The public split holds about 125 molecules, so a difference of 0.001 is a fraction of one molecule and can flip on the private split. Prefer the variant whose reasoning you trust over the one with the luckiest public digit.

**Reproducing.** Fork a notebook that already carries the inputs of the public 0.466 recipe, replace its cells with this notebook, keep **GPU T4**, **internet off** and the **pinned Python 3.12 environment**, then *Save Version → Save & Run All* and submit `submission.csv`. Required inputs: the competition data; `dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets`, `casmi26-pubchem-popularity-prior`, `casmi26-natural-product-panel-benchmark`, `casmi26-natural-product-knowledge-table` and `casmi26-fold-safe-fpnet` at the versions pinned by the 0.466 notebooks; and `sanpier/casmi-spectrum-model-v1`.

In [ ]:
# ── Receipt ─────────────────────────────────────────────────────────────────────────────────
receipt = dict(
    notebook='From Peaks to Structures · leader-safe consensus', submitted=SUBMITTED, variant_requested=VARIANT,
    scoring_rerun=IS_RERUN, commit_mode=COMMIT_MODE, pipeline_ran=PIPELINE_DONE, neural_ok=NEURAL_OK,
    outsiders_ok=OUTSIDERS_OK, consensus_ok=CONSENSUS_OK, frozen_source_sha256=FROZEN_SHA256,
    module_sha256=MODULE_SHA256, variants={k: {x: v[x] for x in ('w_top', 'w_mid', 'rescue', 'changed_rows', 'changed_top1', 'rescued', 'sha256')}
                                           for k, v in REPORT.items()},
    submission_sha256=hashlib.sha256((ROOT / 'submission.csv').read_bytes()).hexdigest(),
    stages=[dict(stage=n, seconds=round(s, 1), status=st) for n, s, st in STAGES],
    total_seconds=round(time.time() - NB_T0, 1))
(ROOT / 'p2s_receipt.json').write_text(json.dumps(receipt, indent=2))
print(json.dumps({k: receipt[k] for k in ('submitted', 'scoring_rerun', 'neural_ok', 'consensus_ok',
                                         'submission_sha256', 'total_seconds')}, indent=2))
for name in ('submission.csv', 'p2s_receipt.json', 'submission_baseline0465.csv', 'submission_public0466.csv'):
    if (ROOT / name).is_file():
        display(FileLink(name))

<a id="credits"></a>
## 7 · Credits and licences

This notebook stands on a public pipeline built in the open by many people. The embedded pipeline notebook keeps its own complete credit section and version history (`p2s_code/frozen_0465.ipynb`); the table below names the main contributions.

| Contribution | Authors |
|---|---|
| Corrected, candidate-specific PubChem promotion (0.465); the spectrum&nbsp;&harr;&nbsp;structure model `casmi-spectrum-model-v1`; the runner, export, neural scorer and fixed-weight fusion of the 0.466 notebooks, embedded here unchanged | [Saner Turhaner (sanpier)](https://www.kaggle.com/code/sanpier/pubchem-neural025-lb-0-466), [candidate-aware promotion](https://www.kaggle.com/code/sanpier/candidate-aware-pubchem-promotion-0-465) |
| Data notebooks and datasets: attributed reproduction assets, PubChem popularity prior, natural-product panel benchmark, fold-safe FPNet, natural-product knowledge table | [Dmitrii Gluzdov](https://www.kaggle.com/code/dmitriigluzdov/casmi-26-from-spectra-to-structures) |
| Complete inference base (E006: window 40, ICEBERG fallback) | [canqiang](https://www.kaggle.com/code/canqiang/casmi26-e006-window-40-ice-fallback-true) |
| Main engine family (v4n, v4m), its models, candidate pool and PubChem tier; ICEBERG and GLACIER packages; fragment-gap score; PubChemLite block | [Ahmed Berat Özer](https://www.kaggle.com/code/ahmedberatozer/casmi26-v4n-inference); `frag_rescore.py` by [Huseyin Emre Aksoy](https://www.kaggle.com/huseyinemreaksoy) (Apache-2.0) |
| Analogue propagation, two-ranker scoring, engine fusion | [prvsiyan](https://www.kaggle.com/code/prvsiyan/analog-propagation-casmi-2026-baseline), [megayak](https://www.kaggle.com/code/megayak/casmi26-two-rankers-one-engine), [seyitkaangunes](https://www.kaggle.com/code/seyitkaangunes/casmi26-v4n-engine-fusion-union-lb-0-399) |
| GLACIER proxies for molecules without [M+H]+, joint ICEBERG and GLACIER re-ordering | [seyitkaangunes](https://www.kaggle.com/code/seyitkaangunes/casmi26-e006-n1a-cc-by-chebi-lb-0-452), [flexonafft](https://www.kaggle.com/code/flexonafft/casmi26-closing-the-evidence-gap) |
| Documentation prior; [M+H]+ restriction for GLACIER | [wangpenghua](https://www.kaggle.com/code/wangpenghua/casmi26-v29-fusion-pop-glmh) |
| Ensemble, promotion and identity audit; metric-identity fusion; ICEBERG fallback | [obstacledeveloper](https://www.kaggle.com/code/obstacledeveloper/casmi26-v1-1-metric-audit-and-safe-tail), [bobthebot369](https://www.kaggle.com/code/bobthebot369/enveda-casmi-2026-v18-sovereign-zenith), [flexonafft](https://www.kaggle.com/code/flexonafft/casmi26-public-baseline-adaptation-experiments), [samanyu1808](https://www.kaggle.com/code/samanyu1808/casmi26-iceberg-where-glacier-is-blind-lb-0-438) |
| Engine 2 rank-2 placement; promotion threshold 150 | [antriksharya](https://www.kaggle.com/code/antriksharya/casmi26-lb-0-443-v7c-engine-2-rank-2-insert), [seyitkaangunes](https://www.kaggle.com/code/seyitkaangunes/casmi26-e006-pubchem-promo-rel-150-lb-0-446) |
| Ranker depth and seeds | [ghazarosbarseghyan91](https://www.kaggle.com/code/ghazarosbarseghyan91/enveda-v7c-e006-fraggap-evgen-v14seeds), [evgendvorkin](https://www.kaggle.com/code/evgendvorkin/enveda-casmi-2026) |
| Candidate catalogues and fingerprints | [sergiomalv](https://www.kaggle.com/datasets/sergiomalv/casmi-chebi-lmsd-structures) (ChEBI, LIPID MAPS), [guhongbin](https://www.kaggle.com/code/guhongbin/casmi26-v447-pubchemlite), [franciscoangulo](https://www.kaggle.com/datasets/franciscoangulo/casmi26-pubchemlite-cc0-candidate-fingerprints), [cosmicbe](https://www.kaggle.com/datasets/cosmicbe/casmi26-pubchemlite), [hengck23](https://www.kaggle.com/datasets/hengck23/hengck23-dreams-enveda-casmi26) (DreaMS resource) |
| Forward spectrum models ICEBERG and GLACIER | [ms-pred](https://github.com/coleygroup/ms-pred) (Coley group) |
| Low-GPU commit fast path (idea) | the public *casmi26 r19-neural025* fork |
| Primary chemistry and spectra | [PubChem](https://pubchem.ncbi.nlm.nih.gov/), [COCONUT](https://coconut.naturalproducts.net/), [LOTUS](https://doi.org/10.7554/eLife.70780), [NPAtlas](https://doi.org/10.1093/nar/gkab941), [ChEBI](https://www.ebi.ac.uk/chebi/), [LIPID MAPS](https://www.lipidmaps.org/), [MassBank](https://massbank.eu/), [MoNA](https://mona.fiehnlab.ucdavis.edu/), [GNPS](https://gnps.ucsd.edu/), [RDKit](https://www.rdkit.org/) |
| Data, task and metric | [Enveda · CASMI 2026](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra), [metric](https://www.kaggle.com/code/metric/casmi-mean-reciprocal-rank) |

**Licences.** The attributed reproduction assets are CC BY-NC-SA 4.0 and every copied folder keeps its source licence (`ATTRIBUTION.md` in that dataset); competition- and COCONUT-derived files are non-commercial with attribution; the other datasets carry the terms stated on their pages. This notebook's own additions (`consensus.py`, `outsiders.py` and the notebook cells) are released under Apache 2.0.

<div style="font-family:system-ui,-apple-system,'Segoe UI',Roboto,sans-serif;margin-top:18px;padding:14px 18px;border-radius:12px;background:#f1effc;color:#52514e;font-size:13.5px;line-height:1.5;">If this notebook helped, an upvote on the notebooks and datasets above helps the people who built the pipeline. Questions and ideas are welcome in the comments.</div>